# Financial Earnings Call NLP Intelligence

## 01 — PDF Extraction

### Objective
Extract raw text from earnings-call transcript PDFs while preserving the original document and page structure.

At this stage, no NLP preprocessing or text cleaning will be performed.

In [1]:
from pathlib import Path

import fitz  # PyMuPDF
import pandas as pd

In [2]:
PROJECT_ROOT = Path("..")
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT.resolve())
print("Raw data folder:", RAW_DATA_DIR.resolve())
print("Exists:", RAW_DATA_DIR.exists())

Project root: D:\financial_earnings_nlp
Raw data folder: D:\financial_earnings_nlp\data\raw
Exists: True


In [3]:
pdf_files = sorted(RAW_DATA_DIR.glob("*.pdf"))

print(f"Number of PDFs found: {len(pdf_files)}")

for pdf in pdf_files:
    print(pdf.name)

Number of PDFs found: 5
august_26.pdf
feb_26.pdf
july_25.pdf
may_26.pdf
nov_25.pdf


## Cell 1 — Open one transcript and inspect basic metadata

In [4]:
sample_pdf = RAW_DATA_DIR / "july_25.pdf"

doc = fitz.open(sample_pdf)

print("File:", sample_pdf.name)
print("Number of pages:", len(doc))
print("PDF metadata:")
print(doc.metadata)

File: july_25.pdf
Number of pages: 24
PDF metadata:
{'format': 'PDF 1.7', 'title': '', 'author': 'admin', 'subject': '', 'keywords': '', 'creator': 'Microsoft® Word for Microsoft 365', 'producer': 'Microsoft® Word for Microsoft 365', 'creationDate': "D:20250729172926+05'30'", 'modDate': "D:20250729183715+05'30'", 'trapped': '', 'encryption': None}


## Cell 2 — Inspect actual raw text

In [5]:
first_page_text = doc[0].get_text("text")

middle_page_index = len(doc) // 2
middle_page_text = doc[middle_page_index].get_text("text")

print("========== FIRST PAGE ==========\n")
print(first_page_text[:3000])

print("\n\n========== MIDDLE PAGE ==========\n")
print(middle_page_text[:3000])

========== FIRST PAGE ==========

 
 
 
Date: July 29, 2025 
To,  
Listing Department 
National Stock Exchange of India Limited 
Exchange Plaza, C-1, G Block, Bandra Kurla Complex, Bandra 
(East), Mumbai - 400 051. 
Symbol: SYRMA 
 
Department of Corporate Service 
BSE Limited 
Phiroze Jeejeebhoy Towers, 
Dalal Street, Mumbai - 400 001. 
Scrip Code: 543573 
Subject: Earnings Call transcript of the Investor Conference held for the unaudited Financial 
Results (Consolidated and Standalone) of the Company for the quarter ended June 30, 2025. 
 
 
Dear Sir/ Madam, 
 
Pursuant to Regulation 30 of the SEBI (Listing Obligations and Disclosure Requirements) Regulations, 
2015, please find attached the transcript in respect to the Earning Conference Call on the unaudited 
Financial Results (Consolidated and Standalone) of the Company for the quarter ended June 30, 2025, held 
on Thursday, July 24, 2025, at 10:30 AM (IST).  
 
The transcript of the conference call will also be accessed at the we

## Cell 3 — Convert this PDF into a page-level DataFrame

In [6]:
page_records = []

for page_num, page in enumerate(doc, start=1):
    raw_text = page.get_text("text")
    
    page_records.append({
        "document": sample_pdf.stem,
        "page_number": page_num,
        "raw_text": raw_text,
        "char_count": len(raw_text),
        "word_count": len(raw_text.split()),
        "line_count": len(raw_text.splitlines())
    })

df_sample = pd.DataFrame(page_records)

df_sample.head()

,document,page_number,raw_text,char_count,word_count,line_count
0,july_25,1,"\n \n \nDate: July 29, 2025 \nTo, \nListing ...",1572,202,55
1,july_25,2,\n \nPage 1 of 23 \n \n \n \n“Syrma SGS Techn...,571,79,37
2,july_25,3,"\nSyrma SGS Technology Limited \nJuly 24, 202...",2607,446,43
3,july_25,4,"\nSyrma SGS Technology Limited \nJuly 24, 202...",2949,527,42
4,july_25,5,"\nSyrma SGS Technology Limited \nJuly 24, 202...",2375,403,37


## Cell 4 — Inspect the DataFrame

In [7]:
df_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   document     24 non-null     str  
 1   page_number  24 non-null     int64
 2   raw_text     24 non-null     str  
 3   char_count   24 non-null     int64
 4   word_count   24 non-null     int64
 5   line_count   24 non-null     int64
dtypes: int64(4), str(2)
memory usage: 1.3 KB


In [8]:
df_sample[
    ["char_count", "word_count", "line_count"]
].describe()

,char_count,word_count,line_count
count,24.000000,24.000000,24.000000
mean,2659.666667,464.541667,47.000000
std,666.735634,129.417417,6.659416
min,571.000000,79.000000,23.000000
25%,2606.250000,458.750000,47.000000
50%,2853.500000,506.500000,49.000000
75%,3032.500000,540.500000,50.000000
max,3265.000000,609.000000,55.000000


In [9]:
df_sample.head()

,document,page_number,raw_text,char_count,word_count,line_count
0,july_25,1,"\n \n \nDate: July 29, 2025 \nTo, \nListing ...",1572,202,55
1,july_25,2,\n \nPage 1 of 23 \n \n \n \n“Syrma SGS Techn...,571,79,37
2,july_25,3,"\nSyrma SGS Technology Limited \nJuly 24, 202...",2607,446,43
3,july_25,4,"\nSyrma SGS Technology Limited \nJuly 24, 202...",2949,527,42
4,july_25,5,"\nSyrma SGS Technology Limited \nJuly 24, 202...",2375,403,37


## Cell 5 — Check suspicious pages

In [10]:
print("Empty pages:", (df_sample["char_count"] == 0).sum())

print("\nPages with the least text:")
display(
    df_sample
    .nsmallest(5, "char_count")
    [["page_number", "char_count", "word_count", "line_count"]]
)

Empty pages: 0

Pages with the least text:


,page_number,char_count,word_count,line_count
1,2,571,79,37
23,24,1148,194,23
0,1,1572,202,55
8,9,2309,398,47
4,5,2375,403,37


# Block 2 — Understand transcript boundaries and structure


## Cell 1 — Inspect the beginning and end of the actual document

In [11]:
# Inspect pages around the transcript beginning and ending

pages_to_check = [1, 2, 3, len(doc) - 2, len(doc) - 1]

for page_index in pages_to_check:
    print("\n" + "=" * 80)
    print(f"PDF PAGE {page_index + 1}")
    print("=" * 80)
    print(doc[page_index].get_text("text")[:2500])


PDF PAGE 2
 
 
Page 1 of 23 
 
 
 
“Syrma SGS Technology Limited Q1 FY '26 Earnings 
Conference Call” 
 
July 24, 2025 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
MANAGEMENT: MR. J. S. GUJRAL – MANAGING DIRECTOR, SYRMA 
SGS TECHNOLOGY LIMITED 
MR. JAYESH DOSHI – DIRECTOR, SYRMA SGS 
TECHNOLOGY LIMITED 
MR. SATENDRA SINGH – CHIEF EXECUTIVE OFFICER, 
SYRMA SGS TECHNOLOGY LIMITED 
MR. BIJAY AGRAWAL – CHIEF FINANCIAL OFFICER, 
SYRMA SGS TECHNOLOGY LIMITED 
MR. NIKHIL GUPTA – HEAD (INVESTOR RELATIONS), 
SYRMA SGS TECHNOLOGY LIMITED 
MODERATOR: 
MR. ACHAL LOHADE – NUVAMA INSTITUTIONAL 
EQUITIES 


PDF PAGE 3
 
Syrma SGS Technology Limited 
July 24, 2025 
 
 
Page 2 of 23 
Moderator: 
Ladies and gentlemen, good day and welcome to Syrma SGS Q1 FY '26 Earnings Conference 
Call hosted by Nuvama Institutional Equities.  
As a reminder, all participants’ lines will be in the listen-only mode and there will be an 
opportunity for you to ask questions after the presentation concludes. Should you need assistance 


In [12]:
# Check how frequently common transcript markers occur

markers = [
    "Moderator:",
    "Operator:",
    "Question:",
    "Disclaimer",
    "Page ",
]

full_sample_text = "\n".join(df_sample["raw_text"])

for marker in markers:
    count = full_sample_text.lower().count(marker.lower())
    print(f"{marker:<15} : {count}")

Moderator:      : 18
Operator:       : 0
Question:       : 0
Disclaimer      : 1
Page            : 23


In [13]:
all_page_records = []

for pdf_path in pdf_files:
    doc_current = fitz.open(pdf_path)

    for page_num, page in enumerate(doc_current, start=1):
        raw_text = page.get_text("text")

        all_page_records.append({
            "document": pdf_path.stem,
            "page_number": page_num,
            "raw_text": raw_text,
            "char_count": len(raw_text),
            "word_count": len(raw_text.split()),
            "line_count": len(raw_text.splitlines())
        })

    doc_current.close()

df_pages = pd.DataFrame(all_page_records)

print("Total pages:", len(df_pages))
print("Total documents:", df_pages["document"].nunique())

df_pages.head()

Total pages: 107
Total documents: 5


,document,page_number,raw_text,char_count,word_count,line_count
0,august_26,1,"\n \n \nDate: August 04, 2026 \n \nTo, \nLis...",1798,225,60
1,august_26,2,\n \nPage 1 of 27 \n \n \n \n“Syrma SGS Techn...,580,86,35
2,august_26,3,"\nSyrma SGS Technology Limited \nJuly 30, 202...",2644,433,47
3,august_26,4,"\nSyrma SGS Technology Limited \nJuly 30, 202...",3076,538,51
4,august_26,5,"\nSyrma SGS Technology Limited \nJuly 30, 202...",2804,455,48


In [14]:
document_summary = (
    df_pages
    .groupby("document")
    .agg(
        pages=("page_number", "count"),
        total_characters=("char_count", "sum"),
        total_words=("word_count", "sum"),
        avg_words_per_page=("word_count", "mean")
    )
    .reset_index()
)

document_summary

,document,pages,total_characters,total_words,avg_words_per_page
0,august_26,28,73896,12712,454.000000
1,feb_26,18,49963,8528,473.777778
2,july_25,24,63832,11149,464.541667
3,may_26,20,57471,9815,490.750000
4,nov_25,17,49862,8584,504.941176


In [15]:
df_pages.nsmallest(
    15,
    "char_count"
)[
    ["document", "page_number", "char_count", "word_count"]
]

,document,page_number,char_count,word_count
71,may_26,2,563,83
29,feb_26,2,569,84
47,july_25,2,571,79
91,nov_25,2,575,85
1,august_26,2,580,86
69,july_25,24,1148,194
45,feb_26,18,1409,247
90,nov_25,1,1518,197
89,may_26,20,1540,268
46,july_25,1,1572,202


# Block 3 — Build the transcript dataset

## Cell 1 — Extract transcript pages

In [16]:
transcript_records = []

for pdf_path in pdf_files:
    doc_current = fitz.open(pdf_path)

    for pdf_page_num, page in enumerate(doc_current, start=1):

        # Physical PDF page 1 = exchange/listing cover letter
        if pdf_page_num == 1:
            continue

        raw_text = page.get_text("text")

        transcript_records.append({
            "document": pdf_path.stem,
            "pdf_page_number": pdf_page_num,
            "raw_text": raw_text,
            "char_count": len(raw_text),
            "word_count": len(raw_text.split()),
            "line_count": len(raw_text.splitlines())
        })

    doc_current.close()

df_transcript_pages = pd.DataFrame(transcript_records)

print("Transcript pages:", len(df_transcript_pages))
print("Documents:", df_transcript_pages["document"].nunique())

df_transcript_pages.head()

Transcript pages: 102
Documents: 5


,document,pdf_page_number,raw_text,char_count,word_count,line_count
0,august_26,2,\n \nPage 1 of 27 \n \n \n \n“Syrma SGS Techn...,580,86,35
1,august_26,3,"\nSyrma SGS Technology Limited \nJuly 30, 202...",2644,433,47
2,august_26,4,"\nSyrma SGS Technology Limited \nJuly 30, 202...",3076,538,51
3,august_26,5,"\nSyrma SGS Technology Limited \nJuly 30, 202...",2804,455,48
4,august_26,6,"\nSyrma SGS Technology Limited \nJuly 30, 202...",2597,407,45


## Cell 2 — Add transcript-relative page number

In [17]:
df_transcript_pages["transcript_page_number"] = (
    df_transcript_pages
    .groupby("document")
    .cumcount() + 1
)

df_transcript_pages[
    [
        "document",
        "pdf_page_number",
        "transcript_page_number",
        "word_count"
    ]
].head(10)

,document,pdf_page_number,transcript_page_number,word_count
0,august_26,2,1,86
1,august_26,3,2,433
2,august_26,4,3,538
3,august_26,5,4,455
4,august_26,6,5,407
5,august_26,7,6,420
6,august_26,8,7,527
7,august_26,9,8,486
8,august_26,10,9,524
9,august_26,11,10,539


## Cell 3 — Verify all five transcripts

In [18]:
transcript_summary = (
    df_transcript_pages
    .groupby("document")
    .agg(
        transcript_pages=("transcript_page_number", "max"),
        total_characters=("char_count", "sum"),
        total_words=("word_count", "sum"),
        avg_words_per_page=("word_count", "mean")
    )
    .reset_index()
)

transcript_summary

,document,transcript_pages,total_characters,total_words,avg_words_per_page
0,august_26,27,72098,12487,462.481481
1,feb_26,17,47990,8295,487.941176
2,july_25,23,62260,10947,475.956522
3,may_26,19,55642,9586,504.526316
4,nov_25,16,48344,8387,524.187500


## Cell 4 — Create working text without touching raw text

In [19]:
df_transcript_pages["clean_text"] = df_transcript_pages["raw_text"].copy()

print(df_transcript_pages.columns.tolist())

print("\nRaw text == clean text:",
      (df_transcript_pages["raw_text"] == df_transcript_pages["clean_text"]).all())

['document', 'pdf_page_number', 'raw_text', 'char_count', 'word_count', 'line_count', 'transcript_page_number', 'clean_text']

Raw text == clean text: True


## Cell 5 — Save our first processed dataset

In [20]:
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DATA_DIR / "transcript_pages.csv"

df_transcript_pages.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", output_path.resolve())
print("Rows:", len(df_transcript_pages))
print("Columns:", len(df_transcript_pages.columns))

Saved: D:\financial_earnings_nlp\data\processed\transcript_pages.csv
Rows: 102
Columns: 8


# Block 4 — Inspect preprocessing noise

## Cell 1 — Look at raw lines from a normal transcript page

In [21]:
sample_row = df_transcript_pages[
    (df_transcript_pages["document"] == "july_25") &
    (df_transcript_pages["transcript_page_number"] == 3)
].iloc[0]

sample_lines = sample_row["raw_text"].splitlines()

for i, line in enumerate(sample_lines):
    print(f"{i:02d}: {repr(line)}")

00: ' '
01: 'Syrma SGS Technology Limited '
02: 'July 24, 2025 '
03: ' '
04: ' '
05: 'Page 3 of 23 '
06: 'So, the recalibration of the strategies which the management consciously started executing from '
07: 'Q2 of last year has panned out as we had planned. And we now believe that we are on that cusp '
08: 'of growth and encashing on this platform which we have built over the last four quarters. '
09: 'It is also very heartening to note that my exports during this quarter have gone up from 180 of '
10: "Q1 FY '25 to 232 in Q1 of FY '26, registering a 29% growth. This, despite the looming "
11: 'uncertainty of the tariffs, was being raised by the U.S. government. So, as things pan out and '
12: 'settle down, I am very confident that going forward, we would be able to grow on the solid '
13: 'platform which we have built now.  '
14: 'I am also happy to share with you that we have entered into a joint venture agreement for '
15: 'manufacturing of PCBs. The PCB industry in India, in our c

## Cell 2 — Find lines repeated across many pages

In [22]:
from collections import Counter

line_counter = Counter()

for text in df_transcript_pages["raw_text"]:
    unique_lines = set(
        line.strip()
        for line in text.splitlines()
        if line.strip()
    )

    line_counter.update(unique_lines)

repeated_lines = pd.DataFrame(
    line_counter.items(),
    columns=["line", "page_frequency"]
).sort_values(
    "page_frequency",
    ascending=False
)

repeated_lines.head(30)

,line,page_frequency
14,Syrma SGS Technology Limited,97
25,Moderator:,66
27,J.S. Gujral:,63
113,Bijay Agrawal:,57
0,"July 30, 2026",27
1480,"July 24, 2025",23
2226,"May 12, 2026",19
899,"January 30, 2026",17
2889,"November 11, 2025",16
2917,J. S. Gujral:,12


## Cell 3 — Specifically inspect page-number patterns

In [23]:
import re

page_number_lines = []

for _, row in df_transcript_pages.iterrows():
    for line in row["raw_text"].splitlines():

        line_clean = line.strip()

        if re.fullmatch(
            r"Page\s+\d+\s+of\s+\d+",
            line_clean,
            flags=re.IGNORECASE
        ):
            page_number_lines.append({
                "document": row["document"],
                "pdf_page_number": row["pdf_page_number"],
                "line": line_clean
            })

df_page_numbers = pd.DataFrame(page_number_lines)

print("Page-number lines found:", len(df_page_numbers))

df_page_numbers.head(10)

Page-number lines found: 102


,document,pdf_page_number,line
0,august_26,2,Page 1 of 27
1,august_26,3,Page 2 of 27
2,august_26,4,Page 3 of 27
3,august_26,5,Page 4 of 27
4,august_26,6,Page 5 of 27
5,august_26,7,Page 6 of 27
6,august_26,8,Page 7 of 27
7,august_26,9,Page 8 of 27
8,august_26,10,Page 9 of 27
9,august_26,11,Page 10 of 27


## Cell 4 — Inspect likely headers by document

In [24]:
for document in df_transcript_pages["document"].unique():

    print("\n" + "=" * 70)
    print(document.upper())
    print("=" * 70)

    doc_rows = df_transcript_pages[
        df_transcript_pages["document"] == document
    ]

    for _, row in doc_rows.iloc[1:4].iterrows():

        lines = [
            line.strip()
            for line in row["raw_text"].splitlines()
            if line.strip()
        ]

        print(
            f"\nTranscript page {row['transcript_page_number']}:"
        )

        print(lines[:5])


AUGUST_26

Transcript page 2:
['Syrma SGS Technology Limited', 'July 30, 2026', 'Page 2 of 27', 'Moderator:', "Ladies and gentlemen, good day and welcome to the Syrma SGS Technology Q1 FY '27"]

Transcript page 3:
['Syrma SGS Technology Limited', 'July 30, 2026', 'Page 3 of 27', 'Jaidit Singh Brar:', 'Thank you, Mr. Gujral. Good morning, all analysts, and I am delighted to be here. I joined Syrma']

Transcript page 4:
['Syrma SGS Technology Limited', 'July 30, 2026', 'Page 4 of 27', 'and every customer, every component shortage is being minutely monitored at the senior-most', 'level.']

FEB_26

Transcript page 2:
['Syrma SGS Technology Limited', 'January 30, 2026', 'Page 2 of 17', 'Moderator:', "Ladies and gentlemen, good day, and welcome to Syrma SGS Q3 FY '26 Earnings Conference"]

Transcript page 3:
['Syrma SGS Technology Limited', 'January 30, 2026', 'Page 3 of 17', 'Going into some of our bit granular detail, what is satisfying and gives us confidence of', 'maintaining this tempo

# Block 5 — First cleaning function

## Cell 1 — Define structural patterns

In [25]:
import re

PAGE_PATTERN = re.compile(
    r"^Page\s+\d+\s+of\s+\d+$",
    flags=re.IGNORECASE
)

DATE_PATTERN = re.compile(
    r"^(January|February|March|April|May|June|July|August|"
    r"September|October|November|December)"
    r"\s+\d{1,2},\s+\d{4}$",
    flags=re.IGNORECASE
)

COMPANY_HEADER_PATTERN = re.compile(
    r"^Syrma SGS Technology Limited$",
    flags=re.IGNORECASE
)

## Cell 2 — Create the cleaning function

In [26]:
def clean_transcript_page(text):
    lines = text.splitlines()
    cleaned_lines = []

    for line in lines:
        line = line.strip()

        # Remove empty lines
        if not line:
            continue

        # Remove repeated company header
        if COMPANY_HEADER_PATTERN.fullmatch(line):
            continue

        # Remove transcript page numbering
        if PAGE_PATTERN.fullmatch(line):
            continue

        # Remove repeated page-header dates
        if DATE_PATTERN.fullmatch(line):
            continue

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)

## Cell 3 — Apply it without modifying raw_text

In [27]:
df_transcript_pages["clean_text"] = (
    df_transcript_pages["raw_text"]
    .apply(clean_transcript_page)
)

df_transcript_pages[
    ["document", "transcript_page_number", "raw_text", "clean_text"]
].head()

,document,transcript_page_number,raw_text,clean_text
0,august_26,1,\n \nPage 1 of 27 \n \n \n \n“Syrma SGS Techn...,“Syrma SGS Technology Limited Q1 FY '27 Invest...
1,august_26,2,"\nSyrma SGS Technology Limited \nJuly 30, 202...","Moderator:\nLadies and gentlemen, good day and..."
2,august_26,3,"\nSyrma SGS Technology Limited \nJuly 30, 202...","Jaidit Singh Brar:\nThank you, Mr. Gujral. Goo..."
3,august_26,4,"\nSyrma SGS Technology Limited \nJuly 30, 202...","and every customer, every component shortage i..."
4,august_26,5,"\nSyrma SGS Technology Limited \nJuly 30, 202...",Coming to our overall business performance:\nG...


## Cell 4 — Compare before vs after

In [28]:
sample = df_transcript_pages[
    (df_transcript_pages["document"] == "july_25") &
    (df_transcript_pages["transcript_page_number"] == 3)
].iloc[0]

print("=" * 80)
print("BEFORE")
print("=" * 80)
print(sample["raw_text"][:2000])

print("\n" + "=" * 80)
print("AFTER")
print("=" * 80)
print(sample["clean_text"][:2000])

BEFORE
 
Syrma SGS Technology Limited 
July 24, 2025 
 
 
Page 3 of 23 
So, the recalibration of the strategies which the management consciously started executing from 
Q2 of last year has panned out as we had planned. And we now believe that we are on that cusp 
of growth and encashing on this platform which we have built over the last four quarters. 
It is also very heartening to note that my exports during this quarter have gone up from 180 of 
Q1 FY '25 to 232 in Q1 of FY '26, registering a 29% growth. This, despite the looming 
uncertainty of the tariffs, was being raised by the U.S. government. So, as things pan out and 
settle down, I am very confident that going forward, we would be able to grow on the solid 
platform which we have built now.  
I am also happy to share with you that we have entered into a joint venture agreement for 
manufacturing of PCBs. The PCB industry in India, in our considered opinion, is ripe for entry 
of organized players. 
The market is estimated at 

## Cell 5 — Validate that the unwanted structures are gone

In [29]:
remaining_page_numbers = 0
remaining_company_headers = 0
remaining_header_dates = 0

for text in df_transcript_pages["clean_text"]:

    for line in text.splitlines():
        line = line.strip()

        if PAGE_PATTERN.fullmatch(line):
            remaining_page_numbers += 1

        if COMPANY_HEADER_PATTERN.fullmatch(line):
            remaining_company_headers += 1

        if DATE_PATTERN.fullmatch(line):
            remaining_header_dates += 1


print("Remaining page numbers :", remaining_page_numbers)
print("Remaining company headers:", remaining_company_headers)
print("Remaining header dates   :", remaining_header_dates)

Remaining page numbers : 0
Remaining company headers: 0
Remaining header dates   : 0


# Block 6 — Repair PDF line wrapping while preserving speakers

## Cell 1 — Detect likely speaker labels

In [30]:
speaker_counter = Counter()

for text in df_transcript_pages["clean_text"]:
    for line in text.splitlines():
        line = line.strip()

        if re.fullmatch(r"[A-Za-z][A-Za-z.\s'-]{1,50}:", line):
            speaker_counter[line] += 1

df_speakers = pd.DataFrame(
    speaker_counter.items(),
    columns=["speaker_label", "frequency"]
).sort_values(
    "frequency",
    ascending=False
)

print("Unique candidate speaker labels:", len(df_speakers))

df_speakers.head(30)

Unique candidate speaker labels: 56


,speaker_label,frequency
4,J.S. Gujral:,130
6,Bijay Agrawal:,91
1,Moderator:,75
51,J. S. Gujral:,36
22,Praveen Sahay:,17
32,Keshav Lahoti:,14
3,Nikhil Gupta:,10
31,Naushad Chaudhary:,10
28,Satendra Singh:,9
17,Sumant Kumar:,8


## Cell 2 — Inspect hyphenated line endings

In [31]:
hyphen_breaks = []

for _, row in df_transcript_pages.iterrows():

    lines = row["clean_text"].splitlines()

    for i in range(len(lines) - 1):

        current_line = lines[i].strip()
        next_line = lines[i + 1].strip()

        if current_line.endswith("-"):
            hyphen_breaks.append({
                "document": row["document"],
                "page": row["transcript_page_number"],
                "current_line": current_line,
                "next_line": next_line
            })

df_hyphen_breaks = pd.DataFrame(hyphen_breaks)

print("Potential hyphenated line breaks:", len(df_hyphen_breaks))

df_hyphen_breaks.head(20)

Potential hyphenated line breaks: 31


,document,page,current_line,next_line
0,august_26,4,consolidated total revenue for the quarter sto...,year growth.
1,august_26,7,"numbers. So, these are numbers which are based...",time expense has been charged or one-time inco...
2,august_26,10,that 35% plus kind of a revenue growth for the...,"on-quarter, there is a higher intake is what w..."
3,august_26,22,"we had onboarded in '23-'24, '24-'25, and '25-...","'26, they may contribute something in this yea..."
4,august_26,23,is it concentrated in any few verticals like i...,based traction here? And beyond the supply cha...
5,august_26,25,"Apart from that consumer business, our ODM bus...","consumer segment here, which is more of a wate..."
6,august_26,26,"As I said, the management is very, very focuse...",term objective. We have our internal plan for ...
7,august_26,26,And I think one point would highlight the whol...,"level executive looking after ESG. So, that, I..."
8,feb_26,2,Call hosted by Axis Capital Limited. As a remi...,"only mode, and there will be an opportunity fo..."
9,feb_26,3,"sort of 4 cylinders of firing, the auto has gr...","tech by 31%, industrial by 29%, IT/railways be..."


## Cell 3 — Create line-wrapping repair function

In [32]:
SPEAKER_PATTERN = re.compile(
    r"^[A-Za-z][A-Za-z.\s'-]{1,50}:$"
)


def repair_line_wrapping(text):

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    repaired_parts = []

    for line in lines:

        # Preserve speaker boundaries
        if SPEAKER_PATTERN.fullmatch(line):

            if repaired_parts:
                repaired_parts.append("\n")

            repaired_parts.append(line)
            repaired_parts.append("\n")

        else:
            repaired_parts.append(line + " ")

    repaired_text = "".join(repaired_parts)

    # Remove spaces surrounding preserved newlines
    repaired_text = re.sub(r" *\n *", "\n", repaired_text)

    # Collapse repeated spaces
    repaired_text = re.sub(r"[ \t]+", " ", repaired_text)

    # Avoid excessive blank lines
    repaired_text = re.sub(r"\n{2,}", "\n", repaired_text)

    return repaired_text.strip()

## Cell 4 — Create processed_text

In [33]:
df_transcript_pages["processed_text"] = (
    df_transcript_pages["clean_text"]
    .apply(repair_line_wrapping)
)

sample = df_transcript_pages[
    (df_transcript_pages["document"] == "july_25") &
    (df_transcript_pages["transcript_page_number"] == 3)
].iloc[0]

print(sample["processed_text"][:3000])

So, the recalibration of the strategies which the management consciously started executing from Q2 of last year has panned out as we had planned. And we now believe that we are on that cusp of growth and encashing on this platform which we have built over the last four quarters. It is also very heartening to note that my exports during this quarter have gone up from 180 of Q1 FY '25 to 232 in Q1 of FY '26, registering a 29% growth. This, despite the looming uncertainty of the tariffs, was being raised by the U.S. government. So, as things pan out and settle down, I am very confident that going forward, we would be able to grow on the solid platform which we have built now. I am also happy to share with you that we have entered into a joint venture agreement for manufacturing of PCBs. The PCB industry in India, in our considered opinion, is ripe for entry of organized players. The market is estimated at about $5 billion, 90% of which is imported, approximately, and only 10% is made in I

## Cell 5 — Inspect a page containing speaker transitions

In [34]:
speaker_sample = df_transcript_pages[
    (df_transcript_pages["document"] == "july_25") &
    (df_transcript_pages["transcript_page_number"] == 4)
].iloc[0]

print(speaker_sample["processed_text"][:4000])

Starting with the revenue numbers, our consolidated total revenue for the quarter is approximately Rs. 960-odd crores as against Rs. 947 crores in the previous quarter. We have been able to see good demand growth in the Auto and Industrial segments, primarily from a year-on-year basis. The growth has been contributed through strong demand across multiple sectors also. Our export revenue for the quarter is approximately Rs. 233 crores, which is again 25% of our total operating revenue for the quarter. Our ODM revenue for the quarter is about 12-odd percent.
Coming to gross margin:
The gross margin for the quarter is 25%, as against 15.5% for the Q1 of last year. The margin improvement is mainly led by healthy business mix, lower consumer and IT business, which is relatively lower margin business, and again, our continuous efforts on operational efficiencies improvement. The operating EBITDA for the quarter stood at healthy Rs. 96 odd crores with a year-on-year growth of 75% and an opera

## Block 7 — Improve structural normalization

In [35]:
def repair_line_wrapping(text):

    # Normalize encoded line-break artifacts first
    text = text.replace("&#xA;", "\n")

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    repaired_lines = []
    current_text = ""

    for line in lines:

        # Speaker label
        if SPEAKER_PATTERN.fullmatch(line):

            if current_text.strip():
                repaired_lines.append(current_text.strip())

            repaired_lines.append(line)
            current_text = ""

        else:
            # If previous line ended with a hyphen,
            # join directly: year- + on-year -> year-on-year
            if current_text.endswith("-"):
                current_text += line

            else:
                if current_text:
                    current_text += " " + line
                else:
                    current_text = line

    if current_text.strip():
        repaired_lines.append(current_text.strip())

    return "\n".join(repaired_lines)

## Cell 2 — Recreate processed_text

In [36]:
df_transcript_pages["processed_text"] = (
    df_transcript_pages["clean_text"]
    .apply(repair_line_wrapping)
)

## Cell 3 — Check for remaining extraction artifacts

In [37]:
checks = {
    "encoded_newline": r"&#xA;",
    "page_number": r"Page\s+\d+\s+of\s+\d+",
    "multiple_spaces": r" {2,}",
}

for name, pattern in checks.items():

    count = (
        df_transcript_pages["processed_text"]
        .str.contains(pattern, regex=True, na=False)
        .sum()
    )

    print(f"{name:<20}: {count}")

encoded_newline     : 0
page_number         : 0
multiple_spaces     : 9


## Cell 4 — Inspect speaker structure directly

In [38]:
sample = df_transcript_pages[
    (df_transcript_pages["document"] == "july_25") &
    (df_transcript_pages["transcript_page_number"] == 4)
].iloc[0]

for line in sample["processed_text"].splitlines():
    print(repr(line))

'Starting with the revenue numbers, our consolidated total revenue for the quarter is approximately Rs. 960-odd crores as against Rs. 947 crores in the previous quarter. We have been able to see good demand growth in the Auto and Industrial segments, primarily from a year-on-year basis. The growth has been contributed through strong demand across multiple sectors also. Our export revenue for the quarter is approximately Rs. 233 crores, which is again 25% of our total operating revenue for the quarter. Our ODM revenue for the quarter is about 12-odd percent.'
'Coming to gross margin:'
'The gross margin for the quarter is 25%, as against 15.5% for the Q1 of last year. The margin improvement is mainly led by healthy business mix, lower consumer and IT business, which is relatively lower margin business, and again, our continuous efforts on operational efficiencies improvement. The operating EBITDA for the quarter stood at healthy Rs. 96 odd crores with a year-on-year growth of 75% and an 

## Cell 5 — Validate financial information survived

In [39]:
sample_text = sample["processed_text"]

financial_terms = [
    "Rs.",
    "%",
    "FY",
    "Q1",
    "EBITDA",
    "PBT",
    "PAT",
    "CAPEX",
    "ROCE"
]

for term in financial_terms:
    print(f"{term:<10} -> {term in sample_text}")

Rs.        -> True
%          -> True
FY         -> False
Q1         -> True
EBITDA     -> True
PBT        -> True
PAT        -> True
CAPEX      -> True
ROCE       -> True


# Block 8 — Validate spaces and separate speakers from headings

## Cell 1 — Inspect the 9 multiple-space cases

In [40]:
multi_space_cases = []

for _, row in df_transcript_pages.iterrows():

    if re.search(r" {2,}", row["processed_text"]):

        matches = list(re.finditer(r" {2,}", row["processed_text"]))

        for match in matches:

            start = max(0, match.start() - 60)
            end = min(len(row["processed_text"]), match.end() + 60)

            multi_space_cases.append({
                "document": row["document"],
                "page": row["transcript_page_number"],
                "context": repr(row["processed_text"][start:end])
            })

df_multi_spaces = pd.DataFrame(multi_space_cases)

print("Multiple-space occurrences:", len(df_multi_spaces))

df_multi_spaces

Multiple-space occurrences: 9


,document,page,context
0,august_26,23,'onics on the vehicle and the charging infrast...
1,august_26,24,"' sector, defense portfolio, would continue to..."
2,feb_26,3,'nk it bodes very well for the electronic indu...
3,feb_26,4,"'based on the demand, which we believe would b..."
4,feb_26,5,"'nd IT, which is a relatively lower gross marg..."
5,july_25,23,'global companies which are adhering to the gl...
6,may_26,12,'R453 crores to INR825 crores. Now this is alm...
7,nov_25,4,'from IT and railway sector. Coming to our net...
8,nov_25,7,"'onfusion, lack of clarity, and that has impac..."


## Cell 2 — Examine all colon-ending labels

In [41]:
colon_labels = Counter()

for text in df_transcript_pages["processed_text"]:

    for line in text.splitlines():

        line = line.strip()

        if line.endswith(":") and len(line) <= 80:
            colon_labels[line] += 1


df_colon_labels = pd.DataFrame(
    colon_labels.items(),
    columns=["label", "frequency"]
).sort_values(
    "frequency",
    ascending=False
)

print("Unique colon-ending labels:", len(df_colon_labels))

df_colon_labels.head(50)

Unique colon-ending labels: 56


,label,frequency
4,J.S. Gujral:,130
6,Bijay Agrawal:,91
1,Moderator:,75
51,J. S. Gujral:,36
22,Praveen Sahay:,17
32,Keshav Lahoti:,14
3,Nikhil Gupta:,10
31,Naushad Chaudhary:,10
28,Satendra Singh:,9
17,Sumant Kumar:,8


## Cell 3 — Build a speaker set from what we already discovered

In [42]:
speaker_labels = set(df_speakers["speaker_label"])

print("Number of speaker labels:", len(speaker_labels))

sorted(speaker_labels)[:20]

Number of speaker labels: 56


['Achal Lohade:',
 'Aniruddha Joshi:',
 'Ankur Sharma:',
 'Ankur:',
 'Anupam Goswami:',
 'Archit Shah:',
 'Arshia Khosla:',
 'Bharat Shah:',
 'Bhavik Mehta:',
 'Bhavya Gandhi:',
 'Bhoomika Nair:',
 'Bijay Agrawal:',
 'Coming to PBT for the quarter:',
 'Coming to ROCE performance for the quarter:',
 'Coming to gross margin:',
 'Coming to our PCB project update:',
 'Coming to our balance sheet performance:',
 'Coming to our order book visibility:',
 'Coming to our overall business performance:',
 'Coming to our working capital performance:']

## Cell 4 — Classify structural lines

In [43]:
structure_records = []

for _, row in df_transcript_pages.iterrows():

    for line in row["processed_text"].splitlines():

        line = line.strip()

        if not line:
            continue

        if line in speaker_labels:
            line_type = "speaker"

        elif line.endswith(":") and len(line) <= 80:
            line_type = "section_heading"

        else:
            line_type = "content"

        structure_records.append({
            "document": row["document"],
            "transcript_page_number": row["transcript_page_number"],
            "line_type": line_type,
            "text": line
        })


df_structure = pd.DataFrame(structure_records)

df_structure["line_type"].value_counts()

line_type
content    618
speaker    553
Name: count, dtype: int64

## Cell 5 — Look at what the classification produced

In [44]:
print("SPEAKERS\n")

display(
    df_structure[
        df_structure["line_type"] == "speaker"
    ][["text"]]
    .drop_duplicates()
    .head(30)
)

print("\nSECTION HEADINGS\n")

display(
    df_structure[
        df_structure["line_type"] == "section_heading"
    ][["text"]]
    .drop_duplicates()
    .head(30)
)

SPEAKERS



,text
1,MODERATOR:
3,Moderator:
5,Nikhil Kandoi:
7,Nikhil Gupta:
9,J.S. Gujral:
11,Jaidit Singh Brar:
16,Bijay Agrawal:
18,Coming to our overall business performance:
20,Coming to the export numbers:
22,Coming to our order book visibility:



SECTION HEADINGS



,text


## Block 9 — Fix speaker detection properly

## Cell 1 — Identify false speaker labels

In [45]:
heading_starters = (
    "coming to ",
    "moving to ",
    "some bit of ",
)

false_speaker_labels = [
    label
    for label in speaker_labels
    if label.lower().startswith(heading_starters)
]

print("False speaker labels:", len(false_speaker_labels))

for label in sorted(false_speaker_labels):
    print(label)

False speaker labels: 11
Coming to PBT for the quarter:
Coming to ROCE performance for the quarter:
Coming to gross margin:
Coming to our PCB project update:
Coming to our balance sheet performance:
Coming to our order book visibility:
Coming to our overall business performance:
Coming to our working capital performance:
Coming to the export numbers:
Moving to our net debt position:
Some bit of update on the new customer additions:


## Cell 2 — Create corrected speaker set

In [46]:
speaker_labels_clean = {
    label
    for label in speaker_labels
    if not label.lower().startswith(heading_starters)
}

print("Original candidates :", len(speaker_labels))
print("Corrected speakers  :", len(speaker_labels_clean))

Original candidates : 56
Corrected speakers  : 45


## Cell 3 — Normalize multiple spaces

In [47]:
def normalize_spaces(text):
    lines = []

    for line in text.splitlines():
        line = re.sub(r"[ \t]+", " ", line).strip()

        if line:
            lines.append(line)

    return "\n".join(lines)


df_transcript_pages["processed_text"] = (
    df_transcript_pages["processed_text"]
    .apply(normalize_spaces)
)

remaining_multi_spaces = (
    df_transcript_pages["processed_text"]
    .str.contains(r" {2,}", regex=True, na=False)
    .sum()
)

print("Pages with multiple spaces:", remaining_multi_spaces)

Pages with multiple spaces: 0


## Cell 4 — Rebuild structural dataset correctly

In [48]:
structure_records = []

for _, row in df_transcript_pages.iterrows():

    for line in row["processed_text"].splitlines():

        line = line.strip()

        if not line:
            continue

        if line in speaker_labels_clean:
            line_type = "speaker"

        elif line.endswith(":") and len(line) <= 80:
            line_type = "section_heading"

        else:
            line_type = "content"

        structure_records.append({
            "document": row["document"],
            "transcript_page_number": row["transcript_page_number"],
            "line_type": line_type,
            "text": line
        })


df_structure = pd.DataFrame(structure_records)

df_structure["line_type"].value_counts()

line_type
content            618
speaker            542
section_heading     11
Name: count, dtype: int64

## Cell 5 — Validate what became headings

In [49]:
print("SECTION HEADINGS\n")

display(
    df_structure[
        df_structure["line_type"] == "section_heading"
    ][["text"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("\nNumber of unique speakers:",
      df_structure.loc[
          df_structure["line_type"] == "speaker",
          "text"
      ].nunique())

SECTION HEADINGS



,text
0,Coming to our overall business performance:
1,Coming to the export numbers:
2,Coming to our order book visibility:
3,Some bit of update on the new customer additions:
4,Coming to our balance sheet performance:
5,Coming to our PCB project update:
6,Coming to gross margin:
7,Coming to PBT for the quarter:
8,Coming to our working capital performance:
9,Moving to our net debt position:



Number of unique speakers: 45


# Block 10 — Sentence and Word Tokenization with NLTK

## Cell 1 — Set up NLTK tokenizer

In [50]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

from nltk.tokenize import sent_tokenize, word_tokenize

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Cell 2 — Take real text from our transcript

In [51]:
sample_text = (
    df_transcript_pages[
        (df_transcript_pages["document"] == "july_25") &
        (df_transcript_pages["transcript_page_number"] == 4)
    ]
    .iloc[0]["processed_text"]
)

print(sample_text[:1500])

Starting with the revenue numbers, our consolidated total revenue for the quarter is approximately Rs. 960-odd crores as against Rs. 947 crores in the previous quarter. We have been able to see good demand growth in the Auto and Industrial segments, primarily from a year-on-year basis. The growth has been contributed through strong demand across multiple sectors also. Our export revenue for the quarter is approximately Rs. 233 crores, which is again 25% of our total operating revenue for the quarter. Our ODM revenue for the quarter is about 12-odd percent.
Coming to gross margin:
The gross margin for the quarter is 25%, as against 15.5% for the Q1 of last year. The margin improvement is mainly led by healthy business mix, lower consumer and IT business, which is relatively lower margin business, and again, our continuous efforts on operational efficiencies improvement. The operating EBITDA for the quarter stood at healthy Rs. 96 odd crores with a year-on-year growth of 75% and an opera

## Cell 3 — Sentence tokenization

In [52]:
sentences = sent_tokenize(sample_text)

print("Number of sentences:", len(sentences))

for i, sentence in enumerate(sentences[:10], start=1):
    print(f"\nSentence {i}:")
    print(sentence)

Number of sentences: 28

Sentence 1:
Starting with the revenue numbers, our consolidated total revenue for the quarter is approximately Rs.

Sentence 2:
960-odd crores as against Rs.

Sentence 3:
947 crores in the previous quarter.

Sentence 4:
We have been able to see good demand growth in the Auto and Industrial segments, primarily from a year-on-year basis.

Sentence 5:
The growth has been contributed through strong demand across multiple sectors also.

Sentence 6:
Our export revenue for the quarter is approximately Rs.

Sentence 7:
233 crores, which is again 25% of our total operating revenue for the quarter.

Sentence 8:
Our ODM revenue for the quarter is about 12-odd percent.

Sentence 9:
Coming to gross margin:
The gross margin for the quarter is 25%, as against 15.5% for the Q1 of last year.

Sentence 10:
The margin improvement is mainly led by healthy business mix, lower consumer and IT business, which is relatively lower margin business, and again, our continuous efforts on o

## Cell 4 — Word tokenization on ONE sentencem

In [53]:
financial_sentence = (
    "The gross margin for the quarter is 25%, "
    "as against 15.5% for Q1 of last year."
)

tokens = word_tokenize(financial_sentence)

print(tokens)
print("\nNumber of tokens:", len(tokens))

['The', 'gross', 'margin', 'for', 'the', 'quarter', 'is', '25', '%', ',', 'as', 'against', '15.5', '%', 'for', 'Q1', 'of', 'last', 'year', '.']

Number of tokens: 20


## Cell 5 — Compare several finance-specific expressions

In [54]:
finance_examples = [
    "Revenue grew 29% year-on-year.",
    "EBITDA margin improved to 10%.",
    "PAT was Rs. 50 crores.",
    "The market is estimated at about $5 billion.",
    "Q1 FY '26 revenue was approximately Rs. 960-odd crores.",
    "ROCE was around 14.5%."
]

for text in finance_examples:
    print("\nTEXT:")
    print(text)

    print("TOKENS:")
    print(word_tokenize(text))


TEXT:
Revenue grew 29% year-on-year.
TOKENS:
['Revenue', 'grew', '29', '%', 'year-on-year', '.']

TEXT:
EBITDA margin improved to 10%.
TOKENS:
['EBITDA', 'margin', 'improved', 'to', '10', '%', '.']

TEXT:
PAT was Rs. 50 crores.
TOKENS:
['PAT', 'was', 'Rs', '.', '50', 'crores', '.']

TEXT:
The market is estimated at about $5 billion.
TOKENS:
['The', 'market', 'is', 'estimated', 'at', 'about', '$', '5', 'billion', '.']

TEXT:
Q1 FY '26 revenue was approximately Rs. 960-odd crores.
TOKENS:
['Q1', 'FY', "'", '26', 'revenue', 'was', 'approximately', 'Rs', '.', '960-odd', 'crores', '.']

TEXT:
ROCE was around 14.5%.
TOKENS:
['ROCE', 'was', 'around', '14.5', '%', '.']


# Block 11 — NLTK vs spaCy

## Cell 1 — Load spaCy

In [55]:
import spacy

nlp = spacy.load("en_core_web_sm")

## Cell 2 — Same transcript, spaCy sentence tokenization

In [56]:
doc_spacy = nlp(sample_text)

spacy_sentences = [sent.text.strip() for sent in doc_spacy.sents]

print("NLTK sentences :", len(sentences))
print("spaCy sentences :", len(spacy_sentences))

for i, sentence in enumerate(spacy_sentences[:10], start=1):
    print(f"\nSentence {i}:")
    print(sentence)

NLTK sentences : 28
spaCy sentences : 21

Sentence 1:
Starting with the revenue numbers, our consolidated total revenue for the quarter is approximately Rs. 960-odd crores as against Rs. 947 crores in the previous quarter.

Sentence 2:
We have been able to see good demand growth in the Auto and Industrial segments, primarily from a year-on-year basis.

Sentence 3:
The growth has been contributed through strong demand across multiple sectors also.

Sentence 4:
Our export revenue for the quarter is approximately Rs. 233 crores, which is again 25% of our total operating revenue for the quarter.

Sentence 5:
Our ODM revenue for the quarter is about 12-odd percent.

Sentence 6:
Coming to gross margin:
The gross margin for the quarter is 25%, as against 15.5% for the Q1 of last year.

Sentence 7:
The margin improvement is mainly led by healthy business mix, lower consumer and IT business, which is relatively lower margin business, and again, our continuous efforts on operational efficiencies

## Cell 3 — Compare tokenization

In [57]:
finance_examples = [
    "PAT was Rs. 50 crores.",
    "Revenue grew 29% year-on-year.",
    "The market is estimated at about $5 billion.",
    "Q1 FY '26 revenue was approximately Rs. 960-odd crores.",
    "ROCE was around 14.5%."
]

for text in finance_examples:

    doc = nlp(text)

    print("\nTEXT:")
    print(text)

    print("spaCy:")
    print([token.text for token in doc])

    print("NLTK:")
    print(word_tokenize(text))


TEXT:
PAT was Rs. 50 crores.
spaCy:
['PAT', 'was', 'Rs', '.', '50', 'crores', '.']
NLTK:
['PAT', 'was', 'Rs', '.', '50', 'crores', '.']

TEXT:
Revenue grew 29% year-on-year.
spaCy:
['Revenue', 'grew', '29', '%', 'year', '-', 'on', '-', 'year', '.']
NLTK:
['Revenue', 'grew', '29', '%', 'year-on-year', '.']

TEXT:
The market is estimated at about $5 billion.
spaCy:
['The', 'market', 'is', 'estimated', 'at', 'about', '$', '5', 'billion', '.']
NLTK:
['The', 'market', 'is', 'estimated', 'at', 'about', '$', '5', 'billion', '.']

TEXT:
Q1 FY '26 revenue was approximately Rs. 960-odd crores.
spaCy:
['Q1', 'FY', "'", '26', 'revenue', 'was', 'approximately', 'Rs', '.', '960', '-', 'odd', 'crores', '.']
NLTK:
['Q1', 'FY', "'", '26', 'revenue', 'was', 'approximately', 'Rs', '.', '960-odd', 'crores', '.']

TEXT:
ROCE was around 14.5%.
spaCy:
['ROCE', 'was', 'around', '14.5', '%', '.']
NLTK:
['ROCE', 'was', 'around', '14.5', '%', '.']


## Cell 4 — Inspect what spaCy knows about each token

In [58]:
example = "EBITDA margin improved to 10% while PAT increased to Rs. 50 crores."

doc = nlp(example)

for token in doc:
    print(
        f"{token.text:<12}"
        f" lemma={token.lemma_:<12}"
        f" pos={token.pos_:<8}"
        f" is_stop={token.is_stop}"
    )

EBITDA       lemma=ebitda       pos=NOUN     is_stop=False
margin       lemma=margin       pos=NOUN     is_stop=False
improved     lemma=improve      pos=VERB     is_stop=False
to           lemma=to           pos=ADP      is_stop=True
10           lemma=10           pos=NUM      is_stop=False
%            lemma=%            pos=NOUN     is_stop=False
while        lemma=while        pos=SCONJ    is_stop=True
PAT          lemma=PAT          pos=PROPN    is_stop=False
increased    lemma=increase     pos=VERB     is_stop=False
to           lemma=to           pos=ADP      is_stop=True
Rs           lemma=Rs           pos=PROPN    is_stop=False
.            lemma=.            pos=PUNCT    is_stop=False
50           lemma=50           pos=NUM      is_stop=False
crores       lemma=crore        pos=NOUN     is_stop=False
.            lemma=.            pos=PUNCT    is_stop=False


# Block 12 — Actually see these concepts on our transcript

## Cell 1 — Select a few real sentences

In [59]:
sample_sentences = spacy_sentences[:10]

for i, sentence in enumerate(sample_sentences, start=1):
    print(f"{i}. {sentence}\n")

1. Starting with the revenue numbers, our consolidated total revenue for the quarter is approximately Rs. 960-odd crores as against Rs. 947 crores in the previous quarter.

2. We have been able to see good demand growth in the Auto and Industrial segments, primarily from a year-on-year basis.

3. The growth has been contributed through strong demand across multiple sectors also.

4. Our export revenue for the quarter is approximately Rs. 233 crores, which is again 25% of our total operating revenue for the quarter.

5. Our ODM revenue for the quarter is about 12-odd percent.

6. Coming to gross margin:
The gross margin for the quarter is 25%, as against 15.5% for the Q1 of last year.

7. The margin improvement is mainly led by healthy business mix, lower consumer and IT business, which is relatively lower margin business, and again, our continuous efforts on operational efficiencies improvement.

8. The operating EBITDA for the quarter stood at healthy Rs.

9. 96 odd crores with a year

## Cell 2 — Inspect tokens + lemma + POS + stopwords

In [60]:
example_sentence = spacy_sentences[0]

doc = nlp(example_sentence)

token_data = []

for token in doc:
    token_data.append({
        "token": token.text,
        "lemma": token.lemma_,
        "pos": token.pos_,
        "is_stop": token.is_stop,
        "is_punct": token.is_punct
    })

df_tokens = pd.DataFrame(token_data)

df_tokens

,token,lemma,pos,is_stop,is_punct
0,Starting,start,VERB,False,False
1,with,with,ADP,True,False
2,the,the,DET,True,False
3,revenue,revenue,NOUN,False,False
4,numbers,number,NOUN,False,False
5,",",",",PUNCT,False,True
6,our,our,PRON,True,False
7,consolidated,consolidated,ADJ,False,False
8,total,total,ADJ,False,False
9,revenue,revenue,NOUN,False,False


## Cell 3 — See what basic preprocessing would do

In [61]:
basic_tokens = [
    token.text.lower()
    for token in doc
    if not token.is_stop
    and not token.is_punct
]

lemma_tokens = [
    token.lemma_.lower()
    for token in doc
    if not token.is_stop
    and not token.is_punct
]

print("ORIGINAL:")
print(example_sentence)

print("\nWITHOUT STOPWORDS/PUNCTUATION:")
print(basic_tokens)

print("\nLEMMATIZED:")
print(lemma_tokens)

ORIGINAL:
Starting with the revenue numbers, our consolidated total revenue for the quarter is approximately Rs. 960-odd crores as against Rs. 947 crores in the previous quarter.

WITHOUT STOPWORDS/PUNCTUATION:
['starting', 'revenue', 'numbers', 'consolidated', 'total', 'revenue', 'quarter', 'approximately', 'rs', '960', 'odd', 'crores', 'rs', '947', 'crores', 'previous', 'quarter']

LEMMATIZED:
['start', 'revenue', 'number', 'consolidated', 'total', 'revenue', 'quarter', 'approximately', 'rs', '960', 'odd', 'crore', 'rs', '947', 'crore', 'previous', 'quarter']


## Cell 4 — Compare stemming vs lemmatization

In [62]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

words = [
    "improve",
    "improved",
    "improving",
    "increase",
    "increased",
    "increasing",
    "companies",
    "growth"
]

print(f"{'WORD':<15} {'STEM':<15} {'LEMMA':<15}")
print("-" * 45)

for word in words:

    spacy_doc = nlp(word)

    lemma = spacy_doc[0].lemma_
    stem = stemmer.stem(word)

    print(f"{word:<15} {stem:<15} {lemma:<15}")

WORD            STEM            LEMMA          
---------------------------------------------
improve         improv          improve        
improved        improv          improve        
improving       improv          improve        
increase        increas         increase       
increased       increas         increase       
increasing      increas         increase       
companies       compani         company        
growth          growth          growth         


# NLP Step 3 — Bag of Words

# Block 13 — Bag of Words

## Cell 1 — Tiny example with CountVectorizer

In [63]:
from sklearn.feature_extraction.text import CountVectorizer

documents = [
    "revenue increased strongly",
    "profit increased strongly",
    "revenue declined"
]

vectorizer = CountVectorizer()

bow_matrix = vectorizer.fit_transform(documents)

print("Vocabulary:")
print(vectorizer.get_feature_names_out())

print("\nBoW Matrix:")
print(bow_matrix.toarray())

Vocabulary:
['declined' 'increased' 'profit' 'revenue' 'strongly']

BoW Matrix:
[[0 1 0 1 1]
 [0 1 1 0 1]
 [1 0 0 1 0]]


## Cell 2 — Make the matrix human-readable

In [64]:
df_bow = pd.DataFrame(
    bow_matrix.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=["Sentence 1", "Sentence 2", "Sentence 3"]
)

df_bow

,declined,increased,profit,revenue,strongly
Sentence 1,0,1,0,1,1
Sentence 2,0,1,1,0,1
Sentence 3,1,0,0,1,0


## Cell 3 — See BoW's word-order limitation

In [65]:
order_examples = [
    "revenue increased profit",
    "profit increased revenue"
]

order_vectorizer = CountVectorizer()

order_matrix = order_vectorizer.fit_transform(order_examples)

print(order_vectorizer.get_feature_names_out())
print(order_matrix.toarray())

['increased' 'profit' 'revenue']
[[1 1 1]
 [1 1 1]]


## Cell 4 — Apply BoW to our five earnings calls

In [66]:
document_texts = (
    df_transcript_pages
    .groupby("document")["processed_text"]
    .apply(" ".join)
)

bow_vectorizer = CountVectorizer(
    lowercase=True,
    stop_words="english"
)

bow_matrix = bow_vectorizer.fit_transform(document_texts)

print("Documents :", bow_matrix.shape[0])
print("Features  :", bow_matrix.shape[1])
print("Matrix shape:", bow_matrix.shape)

Documents : 5
Features  : 3034
Matrix shape: (5, 3034)


## Cell 5 — Most frequent terms in each earnings call

In [67]:
feature_names = bow_vectorizer.get_feature_names_out()

for i, document_name in enumerate(document_texts.index):

    word_counts = bow_matrix[i].toarray().flatten()

    top_indices = word_counts.argsort()[::-1][:15]

    print(f"\n{document_name.upper()}")

    for idx in top_indices:
        print(
            f"{feature_names[idx]:<20}"
            f"{word_counts[idx]}"
        )


AUGUST_26
business            80
quarter             79
year                71
growth              57
think               54
gujral              53
rs                  52
question            50
just                48
bijay               37
years               36
crores              34
sort                30
customers           28
supply              28

FEB_26
crores              113
year                91
quarter             62
growth              57
business            34
margin              32
sort                31
gujral              30
bijay               29
think               28
ebitda              28
industrial          26
question            25
elcome              24
performance         24

JULY_25
year                73
quarter             59
business            57
gujral              51
rs                  43
crores              43
margin              43
question            40
growth              38
bijay               36
line                31
think               30
worki

# NLP Step 4 — N-grams

# Block 14 — N-grams on our transcripts

## Cell 1 — Understand n-grams manually

In [68]:
from nltk.util import ngrams

example_tokens = [
    "operating",
    "ebitda",
    "margin",
    "improved"
]

print("UNIGRAMS:")
print(list(ngrams(example_tokens, 1)))

print("\nBIGRAMS:")
print(list(ngrams(example_tokens, 2)))

print("\nTRIGRAMS:")
print(list(ngrams(example_tokens, 3)))

UNIGRAMS:
[('operating',), ('ebitda',), ('margin',), ('improved',)]

BIGRAMS:
[('operating', 'ebitda'), ('ebitda', 'margin'), ('margin', 'improved')]

TRIGRAMS:
[('operating', 'ebitda', 'margin'), ('ebitda', 'margin', 'improved')]


## Cell 2 — Build unigram + bigram BoW

In [69]:
ngram_vectorizer = CountVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2)
)

ngram_matrix = ngram_vectorizer.fit_transform(document_texts)

print("Unigram-only features :", bow_matrix.shape[1])
print("Unigram + bigram features:", ngram_matrix.shape[1])

Unigram-only features : 3034
Unigram + bigram features: 18742


## Cell 3 — Look specifically at frequent bigrams

In [70]:
ngram_features = ngram_vectorizer.get_feature_names_out()

bigram_indices = [
    i
    for i, feature in enumerate(ngram_features)
    if len(feature.split()) == 2
]

for doc_idx, document_name in enumerate(document_texts.index):

    counts = ngram_matrix[doc_idx].toarray().flatten()

    top_bigram_indices = sorted(
        bigram_indices,
        key=lambda i: counts[i],
        reverse=True
    )[:15]

    print(f"\n{document_name.upper()}")

    for idx in top_bigram_indices:
        if counts[idx] > 0:
            print(f"{ngram_features[idx]:<30} {counts[idx]}")


AUGUST_26
bijay agrawal                  26
order book                     18
supply chain                   17
syrma sgs                      16
comes line                     15
question comes                 15
going forward                  14
moderator question             14
long term                      12
coming years                   10
just add                       9
mr gujral                      9
short term                     9
medtech business               8
second question                8

FEB_26
bijay agrawal                  21
year year                      16
working capital                13
q3 fy                          12
syrma sgs                      11
going forward                  9
inr300 crores                  9
moderator question             9
question line                  9
30 growth                      8
ebitda margin                  8
fy 26                          8
cash flow                      7
crores year                    7
mr gujral

# NLP Step 5 — TF-IDF

# Block 15 — TF-IDF

## Cell 4 — Build TF-IDF representation

In [71]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf_vectorizer.fit_transform(document_texts)

print("Documents:", tfidf_matrix.shape[0])
print("Features :", tfidf_matrix.shape[1])
print("Matrix shape:", tfidf_matrix.shape)

Documents: 5
Features : 18742
Matrix shape: (5, 18742)


## Cell 5 — Highest TF-IDF terms per transcript

In [72]:
tfidf_features = tfidf_vectorizer.get_feature_names_out()

for i, document_name in enumerate(document_texts.index):

    scores = tfidf_matrix[i].toarray().flatten()

    top_indices = scores.argsort()[::-1][:20]

    print(f"\n{document_name.upper()}")

    for idx in top_indices:
        print(
            f"{tfidf_features[idx]:<30}"
            f"{scores[idx]:.4f}"
        )


AUGUST_26
rs                            0.2479
business                      0.2252
quarter                       0.2224
year                          0.1999
growth                        0.1605
think                         0.1520
gujral                        0.1492
question                      0.1408
just                          0.1351
bijay                         0.1042
years                         0.1013
crores                        0.0957
supply                        0.0932
sort                          0.0845
medtech                       0.0810
customers                     0.0788
crore                         0.0763
coming                        0.0760
going                         0.0760
bijay agrawal                 0.0732

FEB_26
crores                        0.3907
year                          0.3146
quarter                       0.2144
growth                        0.1971
business                      0.1176
margin                        0.1106
sort               

# Block 16 — Understanding TF-IDF rather than just using it

## Cell 1 — Document frequency

In [73]:
terms_to_check = [
    "crores",
    "year",
    "quarter",
    "growth",
    "margin",
    "medtech",
    "receivables"
]

for term in terms_to_check:

    if term in tfidf_vectorizer.vocabulary_:

        idx = tfidf_vectorizer.vocabulary_[term]

        document_frequency = (
            tfidf_matrix[:, idx].toarray().flatten() > 0
        ).sum()

        print(
            f"{term:<15} "
            f"appears in {document_frequency}/5 documents"
        )

crores          appears in 5/5 documents
year            appears in 5/5 documents
quarter         appears in 5/5 documents
growth          appears in 5/5 documents
margin          appears in 5/5 documents
medtech         appears in 2/5 documents
receivables     appears in 1/5 documents


## Cell 2 — Inspect IDF directly

In [74]:
for term in terms_to_check:

    if term in tfidf_vectorizer.vocabulary_:

        idx = tfidf_vectorizer.vocabulary_[term]

        print(
            f"{term:<15}"
            f"IDF = {tfidf_vectorizer.idf_[idx]:.4f}"
        )

crores         IDF = 1.0000
year           IDF = 1.0000
quarter        IDF = 1.0000
growth         IDF = 1.0000
margin         IDF = 1.0000
medtech        IDF = 1.6931
receivables    IDF = 2.0986


## Cell 3 — Compare count versus TF-IDF for one transcript

In [75]:
document_name = "august_26"

doc_idx = list(document_texts.index).index(document_name)

for term in terms_to_check:

    if (
        term in bow_vectorizer.vocabulary_
        and term in tfidf_vectorizer.vocabulary_
    ):

        bow_idx = bow_vectorizer.vocabulary_[term]
        tfidf_idx = tfidf_vectorizer.vocabulary_[term]

        count = bow_matrix[doc_idx, bow_idx]
        score = tfidf_matrix[doc_idx, tfidf_idx]

        print(
            f"{term:<15}"
            f"Count = {count:<5}"
            f"TF-IDF = {score:.4f}"
        )

crores         Count = 34   TF-IDF = 0.0957
year           Count = 71   TF-IDF = 0.1999
quarter        Count = 79   TF-IDF = 0.2224
growth         Count = 57   TF-IDF = 0.1605
margin         Count = 25   TF-IDF = 0.0704
medtech        Count = 17   TF-IDF = 0.0810
receivables    Count = 0    TF-IDF = 0.0000


# NLP Step 6 — Task-specific preprocessing

# Block 17 — Build our thematic NLP corpus

## Cell 1 — Remove speaker-label structure

In [76]:
theme_source = (
    df_structure[
        df_structure["line_type"].isin(
            ["content", "section_heading"]
        )
    ]
    .groupby("document")["text"]
    .apply(" ".join)
)

print("Documents:", len(theme_source))

for document, text in theme_source.items():
    print(
        f"{document:<12}"
        f"{len(text):>8} characters"
    )

Documents: 5
august_26      67173 characters
feb_26         44576 characters
july_25        57768 characters
may_26         52006 characters
nov_25         44845 characters


## Cell 2 — Define theme-specific preprocessing

In [77]:
THEME_NOISE = {
    "rs",
    "crore",
    "year",
    "quarter",
    "question",
    "think",
    "just",
    "sort",
    "okay"
}

def prepare_theme_doc(text):

    doc = nlp(text)

    tokens = []

    for token in doc:

        if token.is_space:
            continue

        if token.is_punct:
            continue

        if token.is_stop:
            continue

        if token.like_num:
            continue

        lemma = token.lemma_.lower().strip()

        if not lemma:
            continue

        if lemma in THEME_NOISE:
            continue

        tokens.append(lemma)

    return " ".join(tokens)

## Cell 3 — Apply it and inspect before vs after

In [78]:
theme_processed = {}

for document, text in theme_source.items():
    theme_processed[document] = prepare_theme_doc(text)

theme_processed = pd.Series(theme_processed)

sample_document = "july_25"

print("BEFORE:\n")
print(theme_source[sample_document][:1200])

print("\n" + "=" * 80)

print("\nAFTER:\n")
print(theme_processed[sample_document][:1200])

BEFORE:

“Syrma SGS Technology Limited Q1 FY '26 Earnings Conference Call” MANAGEMENT: MR. J. S. GUJRAL – MANAGING DIRECTOR, SYRMA SGS TECHNOLOGY LIMITED MR. JAYESH DOSHI – DIRECTOR, SYRMA SGS TECHNOLOGY LIMITED MR. SATENDRA SINGH – CHIEF EXECUTIVE OFFICER, MR. BIJAY AGRAWAL – CHIEF FINANCIAL OFFICER, MR. NIKHIL GUPTA – HEAD (INVESTOR RELATIONS), MR. ACHAL LOHADE – NUVAMA INSTITUTIONAL EQUITIES Ladies and gentlemen, good day and welcome to Syrma SGS Q1 FY '26 Earnings Conference Call hosted by Nuvama Institutional Equities. As a reminder, all participants’ lines will be in the listen-only mode and there will be an opportunity for you to ask questions after the presentation concludes. Should you need assistance during the conference call, please signal an operator by pressing ‘*’ then ‘0’ on your touchtone phone. Please note that this conference is being recorded. I now hand the conference over to Mr. Mr. Achal Lohade. Thank you, and over to you, Mr. Lohade. Thank you. Good morning, eve

## Cell 4 — TF-IDF again on our NLP-prepared corpus

In [79]:
theme_tfidf = TfidfVectorizer(
    ngram_range=(1, 2)
)

theme_matrix = theme_tfidf.fit_transform(theme_processed)

theme_features = theme_tfidf.get_feature_names_out()

print("Documents :", theme_matrix.shape[0])
print("Features  :", theme_matrix.shape[1])

for i, document_name in enumerate(theme_processed.index):

    scores = theme_matrix[i].toarray().flatten()

    top_indices = scores.argsort()[::-1][:20]

    print(f"\n{document_name.upper()}")

    for idx in top_indices:
        print(
            f"{theme_features[idx]:<30}"
            f"{scores[idx]:.4f}"
        )

Documents : 5
Features  : 15440

AUGUST_26
business                      0.2800
growth                        0.1878
come                          0.1845
customer                      0.1680
go                            0.1285
grow                          0.1186
number                        0.1153
supply                        0.1087
term                          0.1087
margin                        0.0988
medtech                       0.0948
chain                         0.0935
export                        0.0922
line                          0.0922
revenue                       0.0889
supply chain                  0.0857
order                         0.0857
ramp                          0.0837
kind                          0.0824
segment                       0.0824

FEB_26
growth                        0.2542
export                        0.1873
margin                        0.1739
business                      0.1561
line                          0.1293
elcome                  

# Block 18 — Topic Modeling

## Cell 1 — Prepare topic documents

In [80]:
topic_docs = (
    df_structure[
        df_structure["line_type"] == "content"
    ]["text"]
    .dropna()
)

# Remove extremely tiny conversational fragments
topic_docs = topic_docs[
    topic_docs.str.split().str.len() >= 10
].reset_index(drop=True)

print("Documents for topic modeling:", len(topic_docs))

print("\nExample:")
print(topic_docs.iloc[0][:500])

Documents for topic modeling: 552

Example:
“Syrma SGS Technology Limited Q1 FY '27 Investors Conference Call” MANAGEMENT: MR. J. S. GUJRAL – MANAGING DIRECTOR – SYRMA SGS TECHNOLOGY LIMITED MR. JAYESH DOSHI – WHOLE-TIME DIRECTOR – MR. JAIDIT SINGH BRAR – CHIEF EXECUTIVE OFFICER – SYRMA SGS TECHNOLOGY LIMITED MR. BIJAY AGRAWAL – CHIEF FINANCIAL OFFICER – MR. NIKHIL GUPTA – HEAD, INVESTOR RELATIONS –


## Cell 2 — LDA

In [81]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

lda_vectorizer = CountVectorizer(
    stop_words="english",
    max_df=0.90,
    min_df=2,
    ngram_range=(1, 2)
)

lda_matrix = lda_vectorizer.fit_transform(topic_docs)

lda_model = LatentDirichletAllocation(
    n_components=5,
    random_state=42
)

lda_model.fit(lda_matrix)

print("LDA matrix shape:", lda_matrix.shape)

LDA matrix shape: (552, 3612)


## Cell 3 — Print LDA topics

In [82]:
lda_features = lda_vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(lda_model.components_):

    top_indices = topic.argsort()[-10:][::-1]

    top_words = [
        lda_features[i]
        for i in top_indices
    ]

    print(
        f"Topic {topic_idx + 1}:",
        ", ".join(top_words)
    )

Topic 1: syrma, growth, think, just, syrma sgs, sgs, sir, customers, margin, mr
Topic 2: year, thank, customers, question, quarter, mr, growth, line, forward, gujral
Topic 3: business, capital, working, working capital, days, quarter, margin, growth, just, defense
Topic 4: capex, question, line, pcb, 27, fy, think, project, expect, year
Topic 5: crores, year, quarter, growth, revenue, business, margin, order, ebitda, 30


## Cell 4 — NMF

In [83]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

nmf_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=0.90,
    min_df=2,
    ngram_range=(1, 2)
)

nmf_matrix = nmf_vectorizer.fit_transform(topic_docs)

nmf_model = NMF(
    n_components=5,
    random_state=42
)

nmf_model.fit(nmf_matrix)

nmf_features = nmf_vectorizer.get_feature_names_out()

print("NMF matrix shape:", nmf_matrix.shape)

for topic_idx, topic in enumerate(nmf_model.components_):

    top_indices = topic.argsort()[-10:][::-1]

    top_words = [
        nmf_features[i]
        for i in top_indices
    ]

    print(
        f"Topic {topic_idx + 1}:",
        ", ".join(top_words)
    )

NMF matrix shape: (552, 3612)
Topic 1: business, growth, order, just, margin, book, order book, customers, think, sir
Topic 2: line, question, comes line, question comes, ahead, comes, question line, capital, securities, capital ahead
Topic 3: mr, syrma, sgs, syrma sgs, thank, conference, director, sgs technology, limited, technology limited
Topic 4: crores, year, quarter, rs, revenue, capex, approximately, year year, growth, fy
Topic 5: capital, working, working capital, days, net, net working, 65, cycle, business, capital cycle


# Block 20 — VADER Sentiment

## Cell 1 — Load VADER

In [84]:
import nltk

nltk.download("vader_lexicon")

from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


## Cell 2 — Understand its output first

In [85]:
sentiment_examples = [
    "We have seen strong growth in revenue and profitability.",
    "Demand remained weak and margins declined significantly.",
    "Revenue for the quarter was Rs. 960 crores.",
    "Working capital increased during the quarter."
]

for sentence in sentiment_examples:

    scores = sia.polarity_scores(sentence)

    print("\nTEXT:")
    print(sentence)

    print("SCORES:")
    print(scores)


TEXT:
We have seen strong growth in revenue and profitability.
SCORES:
{'neg': 0.0, 'neu': 0.429, 'pos': 0.571, 'compound': 0.7906}

TEXT:
Demand remained weak and margins declined significantly.
SCORES:
{'neg': 0.468, 'neu': 0.532, 'pos': 0.0, 'compound': -0.5267}

TEXT:
Revenue for the quarter was Rs. 960 crores.
SCORES:
{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}

TEXT:
Working capital increased during the quarter.
SCORES:
{'neg': 0.0, 'neu': 0.704, 'pos': 0.296, 'compound': 0.2732}


## Cell 3 — Build sentence-level dataset from our transcripts

In [86]:
sentiment_records = []

content_rows = df_structure[
    df_structure["line_type"] == "content"
]

for _, row in content_rows.iterrows():

    doc = nlp(row["text"])

    for sentence in doc.sents:

        sentence_text = sentence.text.strip()

        if len(sentence_text.split()) < 5:
            continue

        scores = sia.polarity_scores(sentence_text)

        sentiment_records.append({
            "document": row["document"],
            "sentence": sentence_text,
            "positive": scores["pos"],
            "negative": scores["neg"],
            "neutral": scores["neu"],
            "compound": scores["compound"]
        })

df_sentiment = pd.DataFrame(sentiment_records)

print("Total sentences:", len(df_sentiment))

df_sentiment.head()

Total sentences: 2690


,document,sentence,positive,negative,neutral,compound
0,august_26,“Syrma SGS Technology Limited Q1 FY '27 Invest...,0.000,0.153,0.847,-0.5473
1,august_26,JAIDIT SINGH BRAR – CHIEF EXECUTIVE OFFICER – ...,0.000,0.083,0.917,-0.2263
2,august_26,MR. NIKHIL KANDOI – AXIS CAPITAL LIMITED,0.000,0.275,0.725,-0.2263
3,august_26,"Ladies and gentlemen, good day and welcome to ...",0.212,0.068,0.719,0.6124
4,august_26,"As a reminder, all participant lines will be i...",0.101,0.000,0.899,0.4215


## Cell 4 — Classify + inspect strongest examplesm

In [87]:
def sentiment_label(score):

    if score >= 0.05:
        return "Positive"

    elif score <= -0.05:
        return "Negative"

    else:
        return "Neutral"


df_sentiment["sentiment"] = (
    df_sentiment["compound"]
    .apply(sentiment_label)
)

print(df_sentiment["sentiment"].value_counts())

print("\nTOP POSITIVE SENTENCES")

display(
    df_sentiment
    .nlargest(5, "compound")
    [["document", "sentence", "compound"]]
)

print("\nTOP NEGATIVE SENTENCES")

display(
    df_sentiment
    .nsmallest(5, "compound")
    [["document", "sentence", "compound"]]
)

sentiment
Neutral     1288
Positive    1201
Negative     201
Name: count, dtype: int64

TOP POSITIVE SENTENCES


,document,sentence,compound
272,august_26,But as far as the interest of the management i...,0.9610
1131,july_25,It is my pleasure to share with you that the q...,0.9545
1103,feb_26,And we believe that we are in a good position ...,0.9442
1090,feb_26,"When I say capability building, it is introduc...",0.9424
1087,feb_26,"But on a long-term basis, I think we are very ...",0.9247



TOP NEGATIVE SENTENCES


,document,sentence,compound
2120,may_26,"It takes time for the negativity to settle, in...",-0.7650
755,feb_26,higher industrial and healthcare business and ...,-0.7579
520,august_26,But it is a constant tug of war.,-0.7469
2316,nov_25,"Moving to our debt and treasury position, we h...",-0.7430
2177,may_26,It's a moderate 15%-odd gross material margin ...,-0.7351


# Block 21 — NER + Financial Metric Extraction

## Cell 1 — See spaCy NER on financial text

In [88]:
ner_example = """
Syrma SGS Technology Limited reported revenue of Rs. 960 crores
in Q1 FY26. EBITDA stood at Rs. 96 crores with an EBITDA margin
of 10%. PAT was Rs. 50 crores and export revenue was approximately
Rs. 233 crores.
"""

doc = nlp(ner_example)

for ent in doc.ents:
    print(
        f"{ent.text:<35}"
        f"{ent.label_:<12}"
        f"{spacy.explain(ent.label_)}"
    )

Syrma SGS Technology Limited       ORG         Companies, agencies, institutions, etc.
960                                CARDINAL    Numerals that do not fall under another type
Q1 FY26                            DATE        Absolute or relative dates or periods
96                                 CARDINAL    Numerals that do not fall under another type
10%                                PERCENT     Percentage, including "%"
PAT                                PERSON      People, including fictional
50                                 CARDINAL    Numerals that do not fall under another type
233                                CARDINAL    Numerals that do not fall under another type


## Cell 2 — Run NER on a real transcript sentence

In [89]:
real_example = """
Starting with the revenue numbers, our consolidated total revenue
for the quarter is approximately Rs. 960-odd crores as against
Rs. 947 crores in the previous quarter. Our export revenue for
the quarter is approximately Rs. 233 crores, which is again 25%
of our total operating revenue.
"""

doc_real = nlp(real_example)

for ent in doc_real.ents:
    print(f"{ent.text:<30} → {ent.label_}")

the quarter                    → DATE
960-odd                        → CARDINAL
947                            → CARDINAL
the previous quarter           → DATE
the quarter                    → DATE
233                            → CARDINAL
25%                            → PERCENT


## Cell 3 — Simple financial extraction with regex

In [90]:
import re

financial_text = """
Revenue was approximately Rs. 960 crores.
Export revenue was Rs. 233 crores.
Gross margin was 25%.
Operating EBITDA stood at Rs. 96 crores.
EBITDA margin was 10%.
PBT was Rs. 67.1 crores with a PBT margin of 7%.
PAT was Rs. 50 crores with a PAT margin of 5%.
"""

money_pattern = re.compile(
    r"(?:Rs\.?|INR|₹)\s*"
    r"\d+(?:\.\d+)?"
    r"(?:-odd)?\s*"
    r"(?:crores?|crore|million|billion)",
    flags=re.IGNORECASE
)

percent_pattern = re.compile(
    r"\d+(?:\.\d+)?\s*%"
)

money_values = money_pattern.findall(financial_text)
percentage_values = percent_pattern.findall(financial_text)

print("MONEY:")
print(money_values)

print("\nPERCENTAGES:")
print(percentage_values)

MONEY:
['Rs. 960 crores', 'Rs. 233 crores', 'Rs. 96 crores', 'Rs. 67.1 crores', 'Rs. 50 crores']

PERCENTAGES:
['25%', '10%', '7%', '5%']


## Cell 4 — Extract metric → value pairs

In [91]:
metric_patterns = {
    "Revenue": r"revenue.*?((?:Rs\.?|INR|₹)\s*\d+(?:\.\d+)?(?:-odd)?\s*(?:crores?|crore|million|billion))",

    "Export Revenue": r"export revenue.*?((?:Rs\.?|INR|₹)\s*\d+(?:\.\d+)?(?:-odd)?\s*(?:crores?|crore|million|billion))",

    "Gross Margin": r"gross margin.*?(\d+(?:\.\d+)?\s*%)",

    "EBITDA": r"operating EBITDA.*?((?:Rs\.?|INR|₹)\s*\d+(?:\.\d+)?(?:-odd)?\s*(?:crores?|crore|million|billion))",

    "EBITDA Margin": r"EBITDA margin.*?(\d+(?:\.\d+)?\s*%)",

    "PBT": r"\bPBT\b.*?((?:Rs\.?|INR|₹)\s*\d+(?:\.\d+)?(?:-odd)?\s*(?:crores?|crore|million|billion))",

    "PBT Margin": r"PBT margin.*?(\d+(?:\.\d+)?\s*%)",

    "PAT": r"\bPAT\b.*?((?:Rs\.?|INR|₹)\s*\d+(?:\.\d+)?(?:-odd)?\s*(?:crores?|crore|million|billion))",

    "PAT Margin": r"PAT margin.*?(\d+(?:\.\d+)?\s*%)"
}

results = {}

for metric, pattern in metric_patterns.items():

    match = re.search(
        pattern,
        financial_text,
        flags=re.IGNORECASE
    )

    results[metric] = (
        match.group(1)
        if match
        else None
    )

pd.DataFrame(
    results.items(),
    columns=["metric", "value"]
)

,metric,value
0,Revenue,Rs. 960 crores
1,Export Revenue,Rs. 233 crores
2,Gross Margin,25%
3,EBITDA,Rs. 96 crores
4,EBITDA Margin,10%
5,PBT,Rs. 67.1 crores
6,PBT Margin,7%
7,PAT,Rs. 50 crores
8,PAT Margin,5%


# Block 22 — Classical Extractive Summary

## Cell 1 — Prepare one earnings call

In [92]:
summary_text = document_texts["july_25"]

summary_doc = nlp(summary_text)

summary_sentences = [
    sent.text.strip()
    for sent in summary_doc.sents
    if len(sent.text.split()) >= 8
]

print("Sentences available:", len(summary_sentences))

print("\nExample:")
print(summary_sentences[10])

Sentences available: 521

Example:
I will now hand over the call to Nikhil to take the call forward.


## Cell 2 — Convert sentences to TF-IDF

In [93]:
from sklearn.feature_extraction.text import TfidfVectorizer

summary_vectorizer = TfidfVectorizer(
    stop_words="english"
)

sentence_matrix = summary_vectorizer.fit_transform(
    summary_sentences
)

print("Matrix shape:", sentence_matrix.shape)

Matrix shape: (521, 1326)


## Cell 3 — Score sentences

In [94]:
import numpy as np

non_zero_terms = sentence_matrix.getnnz(axis=1)

sentence_scores = (
    np.asarray(sentence_matrix.sum(axis=1)).ravel()
    / np.maximum(non_zero_terms, 1)
)

top_n = 10

top_indices = sentence_scores.argsort()[::-1][:top_n]

# Put selected sentences back into original transcript order
top_indices = sorted(top_indices)

extractive_summary = [
    summary_sentences[i]
    for i in top_indices
]

## Cell 4 — Show the summary

In [95]:
print("EXTRACTIVE SUMMARY\n")
print("=" * 80)

for i, sentence in enumerate(extractive_summary, start=1):
    print(f"\n{i}. {sentence}")

EXTRACTIVE SUMMARY


1. If we get 60%, it will be 40%.

2. And RDSO has its own pace to move.

3. And you will see it in the coming quarters.

4. And there are three players who will have to absorb this.

5. They would be more for backward integration and all that.

6. Hence, we have not alluded to that in our commentary.

7. And just one last question from my side.

8. And defense is also one of the opportunities.

9. This vertical is a vertical of interest to us.

10. And then there could be again another tick.


## Cell 5 — Centroid-based extractive summary

In [96]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Mean TF-IDF vector of the whole transcript
document_centroid = np.asarray(
    sentence_matrix.mean(axis=0)
)

# Similarity of each sentence to the overall document
centroid_scores = cosine_similarity(
    sentence_matrix,
    document_centroid
).flatten()

top_n = 15

top_indices = centroid_scores.argsort()[::-1][:top_n]

# Restore original transcript order
top_indices = sorted(top_indices)

centroid_summary = [
    summary_sentences[i]
    for i in top_indices
]

## Cell 6 — Display it

In [97]:
print("CENTROID-BASED EXTRACTIVE SUMMARY")
print("=" * 80)

for i, sentence in enumerate(centroid_summary, start=1):
    print(f"\n{i}. {sentence}")

CENTROID-BASED EXTRACTIVE SUMMARY

1. We have been able to see good demand growth in the Auto and Industrial segments, primarily from a year-on-year basis.

2. Our export revenue for the quarter is approximately Rs. 233 crores, which is again 25% of our total operating revenue for the quarter.

3. Coming to gross margin:
The gross margin for the quarter is 25%, as against 15.5% for the Q1 of last year.

4. 96 odd crores with a year-on-year growth of 75% and an operating EBITDA margin of 10%.

5. Coming to PBT for the quarter:
It is Rs. 67.1 crores, again strong year-on-year growth 128% with a PBT margin of 7%.

6. 50 crores, 145% year-on-year growth versus Q1 of last year with a PAT margin of 5%.

7. We expect this to further improve over the year as we expect a higher capacity utilization or maybe the revenue growth as Mr. Gujral has also guided during the all.

8. And if you could just help us again with your guidance for the full year, both on top line and also on margin.

9. And on

# Financial Earnings Call NLP Intelligence
## Classical NLP Phase — Complete Project Summary, Learnings, Failures, Decisions, and Transition to Transformers

---

# 1. Project Objective

The objective of this project is to build an NLP-based **Financial Earnings Call Intelligence system** that can take an earnings-call transcript and extract useful investor-oriented information such as:

- Financial sentiment
- Important financial metrics
- Companies, people, dates and other entities
- Business themes/topics
- Key management commentary
- Important sentences
- Earnings-call summary

The project is intentionally being built in two stages:

1. **Classical NLP**
   - Learn and implement the fundamentals.
   - Understand limitations empirically rather than just theoretically.
   - Establish baseline methods.

2. **Transformer NLP**
   - Use contextual models for financial sentiment, summarization and other downstream tasks.
   - Build the final end-to-end application.

The purpose is not simply to use sophisticated pretrained models.

The goal is to understand:

> Why older NLP approaches work, where they fail, and why transformers improve upon them.

---

# 2. Dataset

The dataset consists of five earnings-call transcript PDFs from Syrma SGS Technology Limited.

Files used:

- July 2025
- November 2025
- February 2026
- May 2026
- August 2026

Overall dataset:

- 5 PDFs
- 107 physical PDF pages
- 102 actual transcript pages after excluding cover pages
- Multiple quarters of management commentary and analyst Q&A

The transcripts contain information such as:

- Revenue
- EBITDA
- PAT/PBT
- Margins
- CAPEX
- Working capital
- Order book
- PCB expansion
- Customer additions
- Export growth
- Segment mix
- Management guidance
- Analyst questions

This makes earnings-call transcripts particularly interesting for NLP because they contain both:

- **structured financial information**
- **unstructured conversational language**

---

# 3. Overall Pipeline Built So Far

```text
PDF Files
   ↓
PDF Text Extraction
   ↓
Page-Level Dataset
   ↓
Structural Cleaning
   ↓
Speaker / Heading / Content Detection
   ↓
Sentence Tokenization
   ↓
Word Tokenization
   ↓
Stopwords / POS / Lemmatization / Stemming
   ↓
Bag of Words
   ↓
N-grams
   ↓
TF-IDF
   ↓
Topic Modeling
   ├── LDA
   └── NMF
   ↓
Classical Sentiment
   └── VADER
   ↓
Named Entity Recognition
   └── spaCy NER
   ↓
Rule-Based Financial Information Extraction
   ↓
Classical Extractive Summarization
   ↓
TRANSFORMERS ← NEXT PHASE

# Block 23 — Word2Vec on our earnings calls

## Cell 1 — Install/import Gensim

%pip install gensim -q

In [98]:
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

import gensim

print("Gensim version:", gensim.__version__)

Gensim version: 4.4.0


## Cell 2 — Build Word2Vec training sentences

In [99]:
word2vec_sentences = []

content_texts = (
    df_structure[
        df_structure["line_type"] == "content"
    ]["text"]
    .dropna()
)

for text in content_texts:

    tokens = simple_preprocess(
        text,
        deacc=True,
        min_len=2
    )

    if len(tokens) >= 3:
        word2vec_sentences.append(tokens)

print("Training sentences:", len(word2vec_sentences))

print("\nExample:")
print(word2vec_sentences[10][:30])

Training sentences: 599

Example:
['growth', 'during', 'the', 'quarter', 'was', 'broad', 'based', 'the', 'consumer', 'business', 'remained', 'our', 'largest', 'vertical', 'with', 'of', 'our', 'business', 'mix', 'and', 'delivered', 'healthy', 'growth', 'supported', 'by', 'faster', 'customer', 'ramp', 'ups', 'and']


## Cell 3 — Train CBOW and Skip-Gram

In [100]:
cbow_model = Word2Vec(
    sentences=word2vec_sentences,
    vector_size=100,
    window=5,
    min_count=2,
    sg=0,
    workers=1,
    epochs=50,
    seed=42
)

skipgram_model = Word2Vec(
    sentences=word2vec_sentences,
    vector_size=100,
    window=5,
    min_count=2,
    sg=1,
    workers=1,
    epochs=50,
    seed=42
)

print("CBOW vocabulary     :", len(cbow_model.wv))
print("Skip-Gram vocabulary:", len(skipgram_model.wv))

CBOW vocabulary     : 1860
Skip-Gram vocabulary: 1860


## Cell 4 — Find semantically related words

In [101]:
test_words = [
    "revenue",
    "growth",
    "margin",
    "capital",
    "customer",
    "pcb"
]

for word in test_words:

    print("\n" + "=" * 70)
    print("WORD:", word)

    if word in cbow_model.wv:

        print("\nCBOW:")
        for similar_word, score in cbow_model.wv.most_similar(
            word,
            topn=5
        ):
            print(
                f"{similar_word:<20}"
                f"{score:.3f}"
            )

        print("\nSKIP-GRAM:")
        for similar_word, score in skipgram_model.wv.most_similar(
            word,
            topn=5
        ):
            print(
                f"{similar_word:<20}"
                f"{score:.3f}"
            )


WORD: revenue

CBOW:
total               0.627
guidance            0.609
consolidated        0.590
implies             0.584
current             0.573

SKIP-GRAM:
came                0.543
concentration       0.528
implies             0.522
stood               0.498
stands              0.497

WORD: growth

CBOW:
rate                0.626
run                 0.623
year                0.560
deliver             0.550
robust              0.549

SKIP-GRAM:
accounts            0.478
expanded            0.474
registering         0.469
base                0.468
fastest             0.463

WORD: margin

CBOW:
margins             0.758
material            0.701
gross               0.684
ebitda              0.633
profit              0.620

SKIP-GRAM:
profile             0.619
expanded            0.601
moderate            0.573
ebitda              0.565
upwards             0.556

WORD: capital

CBOW:
cycle               0.722
loan                0.640
days                0.615
nair                

## Cell 5 — Direct cosine similarity

In [102]:
word_pairs = [
    ("revenue", "growth"),
    ("revenue", "margin"),
    ("working", "capital"),
    ("order", "book"),
    ("customer", "business"),
    ("pcb", "capex")
]

for word1, word2 in word_pairs:

    if (
        word1 in skipgram_model.wv
        and word2 in skipgram_model.wv
    ):

        similarity = skipgram_model.wv.similarity(
            word1,
            word2
        )

        print(
            f"{word1:<12}"
            f"{word2:<12}"
            f"{similarity:.3f}"
        )

revenue     growth      0.439
revenue     margin      0.288
working     capital     0.753
order       book        0.948
customer    business    0.190
pcb         capex       0.452


# Block 24 — Transformer Tokenization + FinBERT

## Cell 1 — Imports

In [103]:
import torch
import transformers

print("PyTorch version     :", torch.__version__)
print("Transformers version:", transformers.__version__)
print("CUDA available      :", torch.cuda.is_available())

PyTorch version     : 2.14.0+cu130
Transformers version: 5.17.0
CUDA available      : True


In [104]:
import torch

print("PyTorch version :", torch.__version__)
print("CUDA runtime    :", torch.version.cuda)
print("CUDA available  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))
    print(
        "VRAM            :",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

PyTorch version : 2.14.0+cu130
CUDA runtime    : 13.0
CUDA available  : True
GPU             : NVIDIA GeForce RTX 3050 Laptop GPU
VRAM            : 4.0 GB


In [107]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA runtime:", torch.version.cuda)

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
CUDA runtime: 13.0


## Cell 2 — Load FinBERT

In [106]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

MODEL_NAME = "ProsusAI/finbert"

finbert_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

finbert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME
)

finbert_model = finbert_model.to(device)
finbert_model.eval()

print("Model loaded on:", next(finbert_model.parameters()).device)

print("\nLabel mapping:")
print(finbert_model.config.id2label)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

NameError: name 'device' is not defined

In [108]:
finbert_model = finbert_model.to(device)
finbert_model.eval()

print(
    "Model loaded on:",
    next(finbert_model.parameters()).device
)

print("\nLabel mapping:")
print(finbert_model.config.id2label)

Model loaded on: cuda:0

Label mapping:
{0: 'positive', 1: 'negative', 2: 'neutral'}


## Cell 3 — See transformer tokenization

In [109]:
text = (
    "Working capital increased during the quarter "
    "while EBITDA margin improved to 10%."
)

tokens = finbert_tokenizer.tokenize(text)

encoded = finbert_tokenizer(
    text,
    return_tensors="pt"
)

token_ids = encoded["input_ids"][0]

tokens_with_special = (
    finbert_tokenizer
    .convert_ids_to_tokens(token_ids)
)

print("ORIGINAL:")
print(text)

print("\nSUBWORD TOKENS:")
print(tokens)

print("\nTOKENS WITH SPECIAL TOKENS:")
print(tokens_with_special)

print("\nINPUT IDS:")
print(token_ids.tolist())

print("\nATTENTION MASK:")
print(encoded["attention_mask"][0].tolist())

ORIGINAL:
Working capital increased during the quarter while EBITDA margin improved to 10%.

SUBWORD TOKENS:
['working', 'capital', 'increased', 'during', 'the', 'quarter', 'while', 'e', '##bit', '##da', 'margin', 'improved', 'to', '10', '%', '.']

TOKENS WITH SPECIAL TOKENS:
['[CLS]', 'working', 'capital', 'increased', 'during', 'the', 'quarter', 'while', 'e', '##bit', '##da', 'margin', 'improved', 'to', '10', '%', '.', '[SEP]']

INPUT IDS:
[101, 2551, 3007, 3445, 2076, 1996, 4284, 2096, 1041, 16313, 2850, 7785, 5301, 2000, 2184, 1003, 1012, 102]

ATTENTION MASK:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## Cell 4 — FinBERT inference function

In [110]:
import torch.nn.functional as F

def finbert_predict(text):

    inputs = finbert_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # Move input tensors to the same device as the model
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = finbert_model(**inputs)

    probabilities = F.softmax(
        outputs.logits,
        dim=-1
    )[0]

    predicted_id = probabilities.argmax().item()
    predicted_label = finbert_model.config.id2label[predicted_id]

    result = {
        "prediction": predicted_label
    }

    for i, probability in enumerate(probabilities):
        label = finbert_model.config.id2label[i]
        result[label] = probability.item()

    return result

## Cell 5 — VADER vs FinBERT test cases

In [111]:
test_sentences = [
    "We have seen strong growth in revenue and profitability.",
    "Demand remained weak and margins declined significantly.",
    "Revenue for the quarter was Rs. 960 crores.",
    "Working capital increased during the quarter.",
    "Debt increased significantly during the quarter.",
    "Operating costs declined significantly during the quarter."
]

for sentence in test_sentences:

    vader_result = sia.polarity_scores(sentence)
    finbert_result = finbert_predict(sentence)

    print("\n" + "=" * 90)

    print("TEXT:")
    print(sentence)

    print(
        "\nVADER:",
        f"compound={vader_result['compound']:.4f}"
    )

    print(
        "FINBERT:",
        finbert_result["prediction"]
    )

    print(
        f"positive : {finbert_result['positive']:.4f}"
    )

    print(
        f"negative : {finbert_result['negative']:.4f}"
    )

    print(
        f"neutral  : {finbert_result['neutral']:.4f}"
    )


TEXT:
We have seen strong growth in revenue and profitability.

VADER: compound=0.7906
FINBERT: positive
positive : 0.9565
negative : 0.0154
neutral  : 0.0281

TEXT:
Demand remained weak and margins declined significantly.

VADER: compound=-0.5267
FINBERT: negative
positive : 0.0087
negative : 0.9724
neutral  : 0.0189

TEXT:
Revenue for the quarter was Rs. 960 crores.

VADER: compound=0.0000
FINBERT: neutral
positive : 0.0485
negative : 0.0350
neutral  : 0.9165

TEXT:
Working capital increased during the quarter.

VADER: compound=0.2732
FINBERT: positive
positive : 0.9592
negative : 0.0170
neutral  : 0.0238

TEXT:
Debt increased significantly during the quarter.

VADER: compound=-0.1027
FINBERT: positive
positive : 0.9557
negative : 0.0216
neutral  : 0.0227

TEXT:
Operating costs declined significantly during the quarter.

VADER: compound=0.0000
FINBERT: negative
positive : 0.0388
negative : 0.9483
neutral  : 0.0129


# Block 25 — FinBERT on the real earnings calls

## Cell 1 — Batch prediction on GPU

In [112]:
def finbert_batch_predict(
    texts,
    batch_size=16,
    max_length=256
):

    predictions = []

    finbert_model.eval()

    for start in range(0, len(texts), batch_size):

        batch_texts = texts[
            start:start + batch_size
        ]

        inputs = finbert_tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.inference_mode():

            outputs = finbert_model(**inputs)

            probs = torch.softmax(
                outputs.logits,
                dim=-1
            )

        for prob in probs:

            predicted_id = prob.argmax().item()

            predictions.append({
                "finbert_sentiment":
                    finbert_model.config.id2label[predicted_id],

                "finbert_positive":
                    prob[0].item(),

                "finbert_negative":
                    prob[1].item(),

                "finbert_neutral":
                    prob[2].item()
            })

    return predictions

## Cell 2 — Run it

In [113]:
texts = df_sentiment["sentence"].tolist()

finbert_results = finbert_batch_predict(
    texts,
    batch_size=16
)

df_finbert = pd.DataFrame(finbert_results)

df_sentiment_compare = pd.concat(
    [
        df_sentiment.reset_index(drop=True),
        df_finbert
    ],
    axis=1
)

df_sentiment_compare.head()

,document,sentence,positive,negative,neutral,compound,sentiment,finbert_sentiment,finbert_positive,finbert_negative,finbert_neutral
0,august_26,“Syrma SGS Technology Limited Q1 FY '27 Invest...,0.000,0.153,0.847,-0.5473,Negative,neutral,0.035575,0.017418,0.947007
1,august_26,JAIDIT SINGH BRAR – CHIEF EXECUTIVE OFFICER – ...,0.000,0.083,0.917,-0.2263,Negative,neutral,0.040240,0.022107,0.937652
2,august_26,MR. NIKHIL KANDOI – AXIS CAPITAL LIMITED,0.000,0.275,0.725,-0.2263,Negative,neutral,0.037214,0.017434,0.945352
3,august_26,"Ladies and gentlemen, good day and welcome to ...",0.212,0.068,0.719,0.6124,Positive,neutral,0.083492,0.012493,0.904014
4,august_26,"As a reminder, all participant lines will be i...",0.101,0.000,0.899,0.4215,Positive,neutral,0.042445,0.015564,0.941991


## Cell 3 — Compare distributions

In [114]:
print("VADER")
print(
    df_sentiment_compare["sentiment"]
    .value_counts()
)

print("\nFINBERT")
print(
    df_sentiment_compare["finbert_sentiment"]
    .value_counts()
)

VADER
sentiment
Neutral     1288
Positive    1201
Negative     201
Name: count, dtype: int64

FINBERT
finbert_sentiment
neutral     1847
positive     721
negative     122
Name: count, dtype: int64


In [115]:
pd.crosstab(
    df_sentiment_compare["sentiment"],
    df_sentiment_compare["finbert_sentiment"],
    margins=True
)

finbert_sentiment,negative,neutral,positive,All
sentiment,,,,
Negative,44,132,25,201
Neutral,42,1062,184,1288
Positive,36,653,512,1201
All,122,1847,721,2690


## Cell 4 — Find disagreement cases

In [117]:
disagreements = df_sentiment_compare[
    df_sentiment_compare["sentiment"].str.lower()
    !=
    df_sentiment_compare["finbert_sentiment"].str.lower()
].copy()

print(
    "Disagreements:",
    len(disagreements),
    "/",
    len(df_sentiment_compare)
)

display(
    disagreements[
        [
            "document",
            "sentence",
            "sentiment",
            "compound",
            "finbert_sentiment",
            "finbert_positive",
            "finbert_negative",
            "finbert_neutral"
        ]
    ].head(50)
)

Disagreements: 1072 / 2690


,document,sentence,sentiment,compound,finbert_sentiment,finbert_positive,finbert_negative,finbert_neutral
0,august_26,“Syrma SGS Technology Limited Q1 FY '27 Invest...,Negative,-0.5473,neutral,0.035575,0.017418,0.947007
1,august_26,JAIDIT SINGH BRAR – CHIEF EXECUTIVE OFFICER – ...,Negative,-0.2263,neutral,0.040240,0.022107,0.937652
2,august_26,MR. NIKHIL KANDOI – AXIS CAPITAL LIMITED,Negative,-0.2263,neutral,0.037214,0.017434,0.945352
3,august_26,"Ladies and gentlemen, good day and welcome to ...",Positive,0.6124,neutral,0.083492,0.012493,0.904014
4,august_26,"As a reminder, all participant lines will be i...",Positive,0.4215,neutral,0.042445,0.015564,0.941991
5,august_26,Should you need assistance during this confere...,Positive,0.3182,neutral,0.040171,0.022223,0.937606
6,august_26,Please note that this conference is being reco...,Positive,0.3182,neutral,0.020214,0.033402,0.946384
7,august_26,I now hand the conference over to Mr. Nikhil K...,Positive,0.4939,neutral,0.055053,0.012305,0.932642
8,august_26,"Thank you and over to you, sir.",Positive,0.3612,neutral,0.195682,0.011076,0.793242
9,august_26,"Nikhil, sir, please go ahead.",Positive,0.3182,neutral,0.048309,0.018680,0.933011


# Block 26 — Transformer Abstractive Summarization

## Cell 1 — Load DistilBART

In [118]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

SUMMARY_MODEL = "sshleifer/distilbart-cnn-12-6"

summary_tokenizer = AutoTokenizer.from_pretrained(
    SUMMARY_MODEL
)

summary_model = AutoModelForSeq2SeqLM.from_pretrained(
    SUMMARY_MODEL
)

summary_model = summary_model.to(device)
summary_model.eval()

print(
    "Summary model device:",
    next(summary_model.parameters()).device
)

config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.22GB            

...

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

Summary model device: cuda:0


## Cell 2 — Create one summarization function

In [124]:
def transformer_summarize(
    text,
    max_input_tokens=800,
    max_summary_tokens=150,
    min_summary_tokens=50
):

    inputs = summary_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_tokens
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():

        summary_ids = summary_model.generate(
            **inputs,
            max_length=max_summary_tokens,
            min_length=min_summary_tokens,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3
        )

    summary = summary_tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary

## Cell 3 — Create the July transcript and chunk it

In [125]:
july_text = " ".join(
    df_structure[
        (df_structure["document"] == "july_25") &
        (df_structure["line_type"] == "content")
    ]["text"]
)

july_doc = nlp(july_text)

july_sentences = [
    sent.text.strip()
    for sent in july_doc.sents
    if sent.text.strip()
]

chunks = []

current_chunk = []
current_tokens = 0

MAX_CHUNK_TOKENS = 750

for sentence in july_sentences:

    sentence_tokens = len(
        summary_tokenizer.encode(
            sentence,
            add_special_tokens=False
        )
    )

    if (
        current_tokens + sentence_tokens
        > MAX_CHUNK_TOKENS
        and current_chunk
    ):

        chunks.append(
            " ".join(current_chunk)
        )

        current_chunk = []
        current_tokens = 0

    current_chunk.append(sentence)
    current_tokens += sentence_tokens

if current_chunk:
    chunks.append(
        " ".join(current_chunk)
    )

print("Transcript sentences:", len(july_sentences))
print("Number of chunks:", len(chunks))

print(
    "First chunk tokens:",
    len(
        summary_tokenizer.encode(
            chunks[0],
            add_special_tokens=False
        )
    )
)

Transcript sentences: 676
Number of chunks: 18
First chunk tokens: 744


## Cell 4 — Summarize every chunk

In [126]:
chunk_summaries = []

for i, chunk in enumerate(chunks, start=1):

    summary = transformer_summarize(
        chunk,
        max_input_tokens=800,
        max_summary_tokens=130,
        min_summary_tokens=40
    )

    chunk_summaries.append(summary)

    print(
        f"Chunk {i}/{len(chunks)} completed"
    )

Chunk 1/18 completed
Chunk 2/18 completed
Chunk 3/18 completed
Chunk 4/18 completed
Chunk 5/18 completed
Chunk 6/18 completed
Chunk 7/18 completed
Chunk 8/18 completed
Chunk 9/18 completed
Chunk 10/18 completed
Chunk 11/18 completed
Chunk 12/18 completed
Chunk 13/18 completed
Chunk 14/18 completed
Chunk 15/18 completed
Chunk 16/18 completed
Chunk 17/18 completed
Chunk 18/18 completed


## Cell 5 — Final hierarchical summary

In [127]:
combined_summaries = " ".join(
    chunk_summaries
)

print(
    "Combined summary words:",
    len(combined_summaries.split())
)

Combined summary words: 1231


In [128]:
final_summary = transformer_summarize(
    combined_summaries,
    max_input_tokens=800,
    max_summary_tokens=220,
    min_summary_tokens=100
)

print("\nTRANSFORMER EXECUTIVE SUMMARY")
print("=" * 90)

print(final_summary)


TRANSFORMER EXECUTIVE SUMMARY
 The recalibration of the strategies which the management consciously started executing from Q2 of last year has panned out as we had planned . Exports during this quarter have gone up from 180 of Q1 FY '25 to 232 in Q1 of FY '26, registering a 29% growth . This, despite the looming uncertainty of the tariffs, was being raised by the U.S. government . The operating EBITDA for the . quarter stood at healthy Rs. 96 odd crores with a year-on-year growth of 75% and an operating E.EBITDA margin of 10% . We have a total gross debt of approximately Rs. 780 crores .


# Block 27 — Proper Hierarchical Summarization

## Cell 1 — Helper to chunk text by transformer tokens

In [129]:
def chunk_text_by_tokens(
    texts,
    tokenizer,
    max_tokens=700
):
    chunks = []
    current_chunk = []
    current_tokens = 0

    for text in texts:

        token_count = len(
            tokenizer.encode(
                text,
                add_special_tokens=False
            )
        )

        if (
            current_tokens + token_count > max_tokens
            and current_chunk
        ):
            chunks.append(
                " ".join(current_chunk)
            )

            current_chunk = []
            current_tokens = 0

        current_chunk.append(text)
        current_tokens += token_count

    if current_chunk:
        chunks.append(
            " ".join(current_chunk)
        )

    return chunks

## Cell 2 — Create second-level chunks

In [130]:
level2_chunks = chunk_text_by_tokens(
    chunk_summaries,
    summary_tokenizer,
    max_tokens=700
)

print("Level 1 summaries:", len(chunk_summaries))
print("Level 2 chunks    :", len(level2_chunks))

for i, chunk in enumerate(level2_chunks, start=1):

    token_count = len(
        summary_tokenizer.encode(
            chunk,
            add_special_tokens=False
        )
    )

    print(
        f"Level 2 chunk {i}: "
        f"{token_count} tokens"
    )

Level 1 summaries: 18
Level 2 chunks    : 3
Level 2 chunk 1: 678 tokens
Level 2 chunk 2: 669 tokens
Level 2 chunk 3: 146 tokens


## Cell 3 — Summarize Level 2

In [131]:
level2_summaries = []

for i, chunk in enumerate(level2_chunks, start=1):

    summary = transformer_summarize(
        chunk,
        max_input_tokens=800,
        max_summary_tokens=180,
        min_summary_tokens=70
    )

    level2_summaries.append(summary)

    print(
        f"Level 2 chunk "
        f"{i}/{len(level2_chunks)} completed"
    )

Level 2 chunk 1/3 completed
Level 2 chunk 2/3 completed
Level 2 chunk 3/3 completed


## Cell 4 — Generate final executive summary

In [132]:
final_input = " ".join(level2_summaries)

final_input_tokens = len(
    summary_tokenizer.encode(
        final_input,
        add_special_tokens=False
    )
)

print(
    "Final input tokens:",
    final_input_tokens
)

final_summary_v2 = transformer_summarize(
    final_input,
    max_input_tokens=800,
    max_summary_tokens=250,
    min_summary_tokens=120
)

print("\nFINAL HIERARCHICAL TRANSFORMER SUMMARY")
print("=" * 90)

print(final_summary_v2)

Final input tokens: 285

FINAL HIERARCHICAL TRANSFORMER SUMMARY
 The recalibration of the strategies which the management consciously started executing from Q2 of last year has panned out as we had planned . Exports during this quarter have gone up from 180 of Q1 FY '25 to 232 in Q1 of FY '26, registering a 29% growth . This, despite the looming uncertainty of the tariffs, was being raised by the U.S. government . The operating EBITDA for the . quarter stood at healthy Rs. 96 odd crores with a year-on-year growth of 75% and an operating E.EBITDA margin of 10% .


# Phase 3 — End-to-End Earnings Call Intelligence

PDF
 ↓
Extract + structural clean
 ↓
┌─────────────────────────────────────────┐
│                                         │
├── Financial Metrics → Regex/rules       │
│                                         │
├── Sentiment → FinBERT                   │
│                                         │
├── Topics → NMF                          │
│                                         │
└── Executive Summary → DistilBART        │
                                          ↓
                         Financial Intelligence Report

                     PDF
                      ↓
             Structural Cleaning
                      ↓
        ┌─────────────┼─────────────┐
        ↓             ↓             ↓
   Regex/Rules     FinBERT       DistilBART
        ↓             ↓             ↓
 Verified Facts    Sentiment     Narrative
        │             │             │
        └─────────────┼─────────────┘
                      ↓
          Earnings Intelligence Report

# Block 28 — Build the final reusable pipeline

## Cell 1 — PDF → clean transcript

In [133]:
def extract_transcript_from_pdf(pdf_path):

    doc = fitz.open(pdf_path)

    pages = []

    for pdf_page_num, page in enumerate(doc, start=1):

        # Earnings-call PDFs in our dataset:
        # physical page 1 = exchange/cover letter
        if pdf_page_num == 1:
            continue

        text = page.get_text("text")

        # Our existing structural cleaning
        text = clean_transcript_page(text)
        text = repair_line_wrapping(text)
        text = normalize_spaces(text)

        pages.append(text)

    doc.close()

    return "\n".join(pages)

## Cell 2 — Financial metric extraction

In [144]:
def extract_financial_metrics(text):

    # Normalize whitespace, but keep the language itself unchanged
    text = re.sub(r"\s+", " ", text)

    MONEY = (
        r"(?:Rs\.?|INR|₹)\s*"
        r"\d+(?:\.\d+)?"
        r"(?:-odd|\s+odd)?\s*"
        r"(?:crores?|crore|million|billion)"
    )

    PERCENT = r"\d+(?:\.\d+)?\s*%"

    metric_patterns = {

        # Require TOTAL / CONSOLIDATED revenue so that
        # export revenue is not accidentally selected.
        "Revenue":
            rf"(?:consolidated\s+total\s+revenue|"
            rf"total\s+revenue|"
            rf"revenue\s+for\s+the\s+quarter)"
            rf"[^.!?]{{0,120}}?({MONEY})",

        "Export Revenue":
            rf"export\s+revenue"
            rf"[^.!?]{{0,100}}?({MONEY})",

        "Gross Margin":
            rf"gross\s+margin"
            rf"[^.!?]{{0,80}}?({PERCENT})",

        # Avoid matching "EBITDA margin" when looking
        # for the EBITDA monetary amount.
        "EBITDA":
            rf"(?:operating\s+)?EBITDA"
            rf"(?!\s+margin)"
            rf"[^.!?]{{0,120}}?({MONEY})",

        # Prefer the explicit current-quarter margin phrase.
        "EBITDA Margin":
            rf"(?:operating\s+)?EBITDA\s+margin"
            rf"\s+(?:of|at|is|was)\s*"
            rf"({PERCENT})",

        "PBT":
            rf"\bPBT\b"
            rf"(?!\s+margin)"
            rf"[^.!?]{{0,120}}?({MONEY})",

        "PBT Margin":
            rf"PBT\s+margin"
            rf"[^.!?]{{0,80}}?({PERCENT})",

        "PAT":
            rf"\bPAT\b"
            rf"(?!\s+margin)"
            rf"[^.!?]{{0,120}}?({MONEY})",

        "PAT Margin":
            rf"PAT\s+margin"
            rf"[^.!?]{{0,80}}?({PERCENT})"
    }

    results = {}

    for metric, pattern in metric_patterns.items():

        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        results[metric] = (
            match.group(1).strip()
            if match
            else None
        )

    return results

## Cell 3 — FinBERT transcript sentiment

In [145]:
def analyze_transcript_sentiment(text):

    doc = nlp(text)

    sentences = [
        sent.text.strip()
        for sent in doc.sents
        if len(sent.text.split()) >= 8
    ]

    results = finbert_batch_predict(
        sentences,
        batch_size=16,
        max_length=256
    )

    sentiment_df = pd.DataFrame(results)
    sentiment_df["sentence"] = sentences

    distribution = (
        sentiment_df["finbert_sentiment"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
        .to_dict()
    )

    return sentiment_df, distribution

## Cell 4 — Reusable transformer summary

In [146]:
def hierarchical_summary(text):

    doc = nlp(text)

    sentences = [
        sent.text.strip()
        for sent in doc.sents
        if sent.text.strip()
    ]

    # LEVEL 1
    level1_chunks = chunk_text_by_tokens(
        sentences,
        summary_tokenizer,
        max_tokens=700
    )

    level1_summaries = []

    for chunk in level1_chunks:

        summary = transformer_summarize(
            chunk,
            max_input_tokens=800,
            max_summary_tokens=130,
            min_summary_tokens=40
        )

        level1_summaries.append(summary)

    # LEVEL 2
    level2_chunks = chunk_text_by_tokens(
        level1_summaries,
        summary_tokenizer,
        max_tokens=700
    )

    level2_summaries = []

    for chunk in level2_chunks:

        summary = transformer_summarize(
            chunk,
            max_input_tokens=800,
            max_summary_tokens=180,
            min_summary_tokens=60
        )

        level2_summaries.append(summary)

    # FINAL
    final_input = " ".join(level2_summaries)

    final_summary = transformer_summarize(
        final_input,
        max_input_tokens=800,
        max_summary_tokens=220,
        min_summary_tokens=90
    )

    return final_summary

## Cell 5 — Final end-to-end function

In [147]:
def analyze_earnings_call(pdf_path):

    print("1. Extracting transcript...")
    transcript = extract_transcript_from_pdf(pdf_path)

    print("2. Extracting financial metrics...")
    metrics = extract_financial_metrics(transcript)

    print("3. Running FinBERT sentiment...")
    sentiment_df, sentiment_distribution = (
        analyze_transcript_sentiment(transcript)
    )

    print("4. Generating transformer summary...")
    executive_summary = hierarchical_summary(transcript)

    result = {
        "file": Path(pdf_path).name,
        "financial_metrics": metrics,
        "sentiment_distribution": sentiment_distribution,
        "executive_summary": executive_summary
    }

    return result, sentiment_df

In [148]:
test_pdf = RAW_DATA_DIR / "july_25.pdf"

report, sentence_sentiment = analyze_earnings_call(
    test_pdf
)

print("\n" + "=" * 90)
print("FINANCIAL EARNINGS CALL INTELLIGENCE REPORT")
print("=" * 90)

print("\nFILE")
print(report["file"])

print("\nFINANCIAL METRICS")
for metric, value in report["financial_metrics"].items():
    print(f"{metric:<20}: {value}")

print("\nFINBERT SENTIMENT DISTRIBUTION")
for label, percentage in report["sentiment_distribution"].items():
    print(f"{label:<10}: {percentage}%")

print("\nEXECUTIVE SUMMARY")
print(report["executive_summary"])

1. Extracting transcript...
2. Extracting financial metrics...
3. Running FinBERT sentiment...
4. Generating transformer summary...

FINANCIAL EARNINGS CALL INTELLIGENCE REPORT

FILE
july_25.pdf

FINANCIAL METRICS
Revenue             : Rs. 960-odd crores
Export Revenue      : Rs. 233 crores
Gross Margin        : 25%
EBITDA              : Rs. 96 odd crores
EBITDA Margin       : 10%
PBT                 : Rs. 67.1 crores
PBT Margin          : 7%
PAT                 : Rs. 50 crores
PAT Margin          : 5%

FINBERT SENTIMENT DISTRIBUTION
neutral   : 70.19%
positive  : 25.19%
negative  : 4.62%

EXECUTIVE SUMMARY
 J.S. Gujral Ji: "The focus from our perspective is customers, obsessed with the customers" EBITDA margins are up from 5.2% of Q1 last year to over 10% in this year . Railways is still a lumpy business, and I expect that this year it should be between Rs. 80 crores and Rs. 100 crores . Industrial smart metering and utility metering is one segment, which is both domestic and export .

# Block 29 — Fix Financial Metric Extraction

## Cell 1 — Local-context financial extraction

In [142]:
def extract_financial_metrics(text):

    # Normalize whitespace, but keep the language itself unchanged
    text = re.sub(r"\s+", " ", text)

    MONEY = (
        r"(?:Rs\.?|INR|₹)\s*"
        r"\d+(?:\.\d+)?"
        r"(?:-odd|\s+odd)?\s*"
        r"(?:crores?|crore|million|billion)"
    )

    PERCENT = r"\d+(?:\.\d+)?\s*%"

    metric_patterns = {

        # Require TOTAL / CONSOLIDATED revenue so that
        # export revenue is not accidentally selected.
        "Revenue":
            rf"(?:consolidated\s+total\s+revenue|"
            rf"total\s+revenue|"
            rf"revenue\s+for\s+the\s+quarter)"
            rf"[^.!?]{{0,120}}?({MONEY})",

        "Export Revenue":
            rf"export\s+revenue"
            rf"[^.!?]{{0,100}}?({MONEY})",

        "Gross Margin":
            rf"gross\s+margin"
            rf"[^.!?]{{0,80}}?({PERCENT})",

        # Avoid matching "EBITDA margin" when looking
        # for the EBITDA monetary amount.
        "EBITDA":
            rf"(?:operating\s+)?EBITDA"
            rf"(?!\s+margin)"
            rf"[^.!?]{{0,120}}?({MONEY})",

        # Prefer the explicit current-quarter margin phrase.
        "EBITDA Margin":
            rf"(?:operating\s+)?EBITDA\s+margin"
            rf"\s+(?:of|at|is|was)\s*"
            rf"({PERCENT})",

        "PBT":
            rf"\bPBT\b"
            rf"(?!\s+margin)"
            rf"[^.!?]{{0,120}}?({MONEY})",

        "PBT Margin":
            rf"PBT\s+margin"
            rf"[^.!?]{{0,80}}?({PERCENT})",

        "PAT":
            rf"\bPAT\b"
            rf"(?!\s+margin)"
            rf"[^.!?]{{0,120}}?({MONEY})",

        "PAT Margin":
            rf"PAT\s+margin"
            rf"[^.!?]{{0,80}}?({PERCENT})"
    }

    results = {}

    for metric, pattern in metric_patterns.items():

        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        results[metric] = (
            match.group(1).strip()
            if match
            else None
        )

    return results

## Cell 2 — Test extraction only

In [143]:
test_pdf = RAW_DATA_DIR / "july_25.pdf"

test_transcript = extract_transcript_from_pdf(
    test_pdf
)

fixed_metrics = extract_financial_metrics(
    test_transcript
)

for metric, value in fixed_metrics.items():
    print(f"{metric:<20}: {value}")

Revenue             : Rs. 960-odd crores
Export Revenue      : Rs. 233 crores
Gross Margin        : 25%
EBITDA              : Rs. 96 odd crores
EBITDA Margin       : 10%
PBT                 : Rs. 67.1 crores
PBT Margin          : 7%
PAT                 : Rs. 50 crores
PAT Margin          : 5%


# Block 30 — Start packaging

## Cell 1 — Create final directories/files

In [149]:
from pathlib import Path

PROJECT_ROOT = Path("..")

SRC_DIR = PROJECT_ROOT / "src"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

SRC_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

(SRC_DIR / "__init__.py").touch()

print("src:", SRC_DIR.resolve())
print("outputs:", OUTPUT_DIR.resolve())

src: D:\financial_earnings_nlp\src
outputs: D:\financial_earnings_nlp\outputs


## Cell 2 — preprocessing.py

In [150]:
%%writefile ../src/preprocessing.py

import re
import fitz


PAGE_PATTERN = re.compile(
    r"^Page\s+\d+\s+of\s+\d+$",
    flags=re.IGNORECASE
)

DATE_PATTERN = re.compile(
    r"^(January|February|March|April|May|June|July|August|"
    r"September|October|November|December)"
    r"\s+\d{1,2},\s+\d{4}$",
    flags=re.IGNORECASE
)

COMPANY_HEADER_PATTERN = re.compile(
    r"^Syrma SGS Technology Limited$",
    flags=re.IGNORECASE
)


def clean_transcript_page(text):
    lines = text.splitlines()
    cleaned_lines = []

    for line in lines:
        line = line.strip()

        if not line:
            continue

        if COMPANY_HEADER_PATTERN.fullmatch(line):
            continue

        if PAGE_PATTERN.fullmatch(line):
            continue

        if DATE_PATTERN.fullmatch(line):
            continue

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)


def normalize_spaces(text):
    lines = []

    for line in text.splitlines():
        line = re.sub(r"[ \t]+", " ", line).strip()

        if line:
            lines.append(line)

    return "\n".join(lines)


def extract_transcript_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)

    pages = []

    for pdf_page_num, page in enumerate(doc, start=1):

        # Dataset convention:
        # first physical page is the exchange cover letter
        if pdf_page_num == 1:
            continue

        text = page.get_text("text")

        text = clean_transcript_page(text)
        text = normalize_spaces(text)

        pages.append(text)

    doc.close()

    return "\n".join(pages)

Writing ../src/preprocessing.py


## Cell 3 — financial_extraction.py

In [168]:
%%writefile ../src/financial_extraction.py

import re


def extract_financial_metrics(text):

    text = re.sub(r"\s+", " ", text)

    # Supports:
    # Rs. 960 crores
    # Rs. 1,604 crores
    # Rs. 67.1 crores
    # Rs. 960-odd crores
    # Rs. 96 odd crores

    MONEY = (
        r"(?:Rs\.?|INR|₹)\s*"
        r"\d[\d,]*(?:\.\d+)?"
        r"(?:-odd|\s+odd)?\s*"
        r"(?:crores?|crore|million|billion)"
    )

    PERCENT = r"\d+(?:\.\d+)?\s*%"

    metric_patterns = {

        "Revenue":
            rf"(?:consolidated\s+total\s+revenue|"
            rf"total\s+revenue|"
            rf"revenue\s+for\s+the\s+quarter)"
            rf"[^.!?]{{0,120}}?({MONEY})",

        "Export Revenue":
            rf"export\s+revenue"
            rf"[^.!?]{{0,100}}?({MONEY})",

        "Gross Margin":
            rf"gross\s+margin"
            rf"[^.!?]{{0,80}}?({PERCENT})",

        "EBITDA":
            rf"(?:operating\s+)?EBITDA"
            rf"(?!\s+margin)"
            rf"[^.!?]{{0,120}}?({MONEY})",

        "EBITDA Margin":
            rf"(?:operating\s+)?EBITDA\s+margin"
            rf"\s+(?:of|at|is|was)\s*"
            rf"({PERCENT})",

        "PBT":
            rf"\bPBT\b"
            rf"(?!\s+margin)"
            rf"[^.!?]{{0,120}}?({MONEY})",

        "PBT Margin":
            rf"PBT\s+margin"
            rf"[^.!?]{{0,80}}?({PERCENT})",

        "PAT":
            rf"\bPAT\b"
            rf"(?!\s+margin)"
            rf"[^.!?]{{0,120}}?({MONEY})",

        "PAT Margin":
            rf"PAT\s+margin"
            rf"[^.!?]{{0,80}}?({PERCENT})"
    }

    results = {}

    for metric, pattern in metric_patterns.items():

        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        results[metric] = (
            match.group(1).strip()
            if match
            else None
        )

    return results

Overwriting ../src/financial_extraction.py


## Cell 4 — Verify modules actually work

In [170]:
import sys

sys.path.append(str(SRC_DIR.resolve()))

from preprocessing import extract_transcript_from_pdf
from financial_extraction import extract_financial_metrics


test_pdf = RAW_DATA_DIR / "july_25.pdf"

module_transcript = extract_transcript_from_pdf(test_pdf)

module_metrics = extract_financial_metrics(
    module_transcript
)

for metric, value in module_metrics.items():
    print(f"{metric:<20}: {value}")

Revenue             : Rs. 960-odd crores
Export Revenue      : Rs. 233 crores
Gross Margin        : 25%
EBITDA              : Rs. 96 odd crores
EBITDA Margin       : 10%
PBT                 : Rs. 67.1 crores
PBT Margin          : 7%
PAT                 : Rs. 50 crores
PAT Margin          : 5%


# Cell 1 — Run August

In [172]:
august_pdf = RAW_DATA_DIR / "august_26.pdf"

august_report, august_sentiment = analyze_earnings_call(
    august_pdf
)

print("\n" + "=" * 90)
print("AUGUST 2026 — FINANCIAL EARNINGS CALL INTELLIGENCE")
print("=" * 90)

print("\nFINANCIAL METRICS")

for metric, value in august_report["financial_metrics"].items():
    print(f"{metric:<20}: {value}")

print("\nFINBERT SENTIMENT DISTRIBUTION")

for label, percentage in august_report["sentiment_distribution"].items():
    print(f"{label:<10}: {percentage}%")

print("\nEXECUTIVE SUMMARY")
print(august_report["executive_summary"])

1. Extracting transcript...
2. Extracting financial metrics...
3. Running FinBERT sentiment...
4. Generating transformer summary...

AUGUST 2026 — FINANCIAL EARNINGS CALL INTELLIGENCE

FINANCIAL METRICS
Revenue             : None
Export Revenue      : None
Gross Margin        : None
EBITDA              : Rs. 162 crores
EBITDA Margin       : None
PBT                 : Rs. 141 crores
PBT Margin          : None
PAT                 : Rs. 106 crores
PAT Margin          : None

FINBERT SENTIMENT DISTRIBUTION
neutral   : 65.59%
positive  : 29.64%
negative  : 4.77%

EXECUTIVE SUMMARY
 The Syrma SGS Technology Q1 FY '27 Investors' Con Call is hosted by Axis Capital Limited . The company will discuss the performance of the company during the 1st Quarter and 2nd Fiscal Year 2027, followed by a detailed question-and-answer session . J.S. Gujral: Demand remains bullish but supply chain is a cause of concern because of the geopolitical situation . Domestically, automotive remains to be the cornersto

In [173]:
august_metrics = extract_financial_metrics(
    august_transcript
)

for metric, value in august_metrics.items():
    print(f"{metric:<20}: {value}")

Revenue             : None
Export Revenue      : None
Gross Margin        : None
EBITDA              : Rs. 162 crores
EBITDA Margin       : None
PBT                 : Rs. 141 crores
PBT Margin          : None
PAT                 : Rs. 106 crores
PAT Margin          : None


## Cell 2 — inspect evidence for anything suspicious

In [174]:
august_transcript = extract_transcript_from_pdf(
    august_pdf
)

terms = [
    "revenue",
    "gross margin",
    "EBITDA",
    "PBT",
    "PAT",
    "working capital"
]

for term in terms:

    print("\n" + "=" * 90)
    print(term.upper())
    print("=" * 90)

    matches = [
        sent.text.strip()
        for sent in nlp(august_transcript).sents
        if term.lower() in sent.text.lower()
    ]

    for sentence in matches[:15]:
        print("•", sentence)


REVENUE
• Starting with the revenue numbers, our
consolidated total revenue for the quarter stood at Rs. 1,604 crores, registering a 67% year-on-
year growth.
• Similarly, the automotive business continued its strong momentum and now contributes about
one-fourth of our overall total revenue with 25% of our business mix.
• However, we remain confident about
the medium-term opportunity on this business, seeing the overall order book visibility..
Coming to the export numbers:
As Mr. Gujral has also highlighted, this has performed very well and contributed approximately
24% of our operating revenue, reflecting our growing participation into the global supply chain.
• On the customer concentration for the quarter, my top 5 customers contributed around 38%,
while top 10 customers contributed around 51% and top 20 around 66% of our total revenue for
the quarter.
• Is there any one-time revenue booking done in Q1?
• So, any postponement of revenue booking or order booking or like
that?
• If n

In [175]:
def get_overall_sentiment(sentiment_df):

    avg_positive = sentiment_df[
        "finbert_positive"
    ].mean()

    avg_negative = sentiment_df[
        "finbert_negative"
    ].mean()

    avg_neutral = sentiment_df[
        "finbert_neutral"
    ].mean()

    scores = {
        "positive": avg_positive,
        "negative": avg_negative,
        "neutral": avg_neutral
    }

    overall_label = max(
        scores,
        key=scores.get
    )

    return {
        "label": overall_label,
        "positive": round(avg_positive * 100, 2),
        "neutral": round(avg_neutral * 100, 2),
        "negative": round(avg_negative * 100, 2)
    }

In [176]:
august_overall_sentiment = get_overall_sentiment(
    august_sentiment
)

august_overall_sentiment

{'label': 'neutral',
 'positive': np.float64(31.63),
 'neutral': np.float64(61.47),
 'negative': np.float64(6.89)}

## Cell 4 — Final July + August validation

In [180]:
validation_files = {
    "July 2025": RAW_DATA_DIR / "july_25.pdf",
    "August 2026": RAW_DATA_DIR / "august_26.pdf"
}

for name, pdf_path in validation_files.items():

    print("\n" + "=" * 90)
    print(name.upper())
    print("=" * 90)

    transcript = extract_transcript_from_pdf(
        pdf_path
    )

    metrics = extract_financial_metrics(
        transcript
    )

    print("\nFINANCIAL METRICS")

    for metric, value in metrics.items():
        print(f"{metric:<20}: {value}")


JULY 2025

FINANCIAL METRICS
Revenue             : Rs. 960-odd crores
Export Revenue      : Rs. 233 crores
Gross Margin        : 25%
EBITDA              : Rs. 96 odd crores
EBITDA Margin       : 10%
PBT                 : Rs. 67.1 crores
PBT Margin          : 7%
PAT                 : Rs. 50 crores
PAT Margin          : 5%

AUGUST 2026

FINANCIAL METRICS
Revenue             : Rs. 1,604 crores
Export Revenue      : None
Gross Margin        : None
EBITDA              : Rs. 162 crores
EBITDA Margin       : None
PBT                 : Rs. 141 crores
PBT Margin          : None
PAT                 : Rs. 106 crores
PAT Margin          : None


In [178]:
import importlib
import financial_extraction

importlib.reload(financial_extraction)

from financial_extraction import extract_financial_metrics

In [179]:
august_transcript = extract_transcript_from_pdf(
    RAW_DATA_DIR / "august_26.pdf"
)

august_metrics = extract_financial_metrics(
    august_transcript
)

for metric, value in august_metrics.items():
    print(f"{metric:<20}: {value}")

Revenue             : Rs. 1,604 crores
Export Revenue      : None
Gross Margin        : None
EBITDA              : Rs. 162 crores
EBITDA Margin       : None
PBT                 : Rs. 141 crores
PBT Margin          : None
PAT                 : Rs. 106 crores
PAT Margin          : None
